In [ ]:
!pip install -U datasets pyarrow
!pip install -q ijson
!pip install bert-score

# NOTE: You MUST restart the runtime (Runtime -> Restart session) after this cell finishes,
# otherwise the old versions of datasets and pyarrow will still be loaded in memory.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 9.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 104.6 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [datasets]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
import os

from datasets import load_dataset


HF_DATASET_ID = "Exploration-Lab/CS-781-Capstone"
hf_token = "HF-TOKEN"


def load_legal_dataset(
    hf_token=None,
    split=None,
    phase=None,
    streaming=False
):

    if hf_token is None:
        raise ValueError(
            "Hugging Face token not found. "
            "Set HF_TOKEN or pass token explicitly."
        )

    dataset = load_dataset(
        HF_DATASET_ID,
        phase,
        split=split,
        token=hf_token,
        streaming=streaming,
    )

    return dataset


In [ ]:
from huggingface_hub import hf_hub_download
import os
from huggingface_hub import login

login()

repo_id = "Exploration-Lab/CS-781-Capstone"

acts_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_acts.json",
    repo_type="dataset",
)

high_courts_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_high_courts.json",
    repo_type="dataset",
)

supreme_court_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_supreme_court.json",
    repo_type="dataset",
)

tribunals_path = hf_hub_download(
    repo_id=repo_id,
    filename="Phase_1_Pretraining/Train/train_tribunals.json",
    repo_type="dataset",
)

print(acts_path)
print(high_courts_path)
print(supreme_court_path)
print(tribunals_path)

Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_acts.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_high_courts.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_supreme_court.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Train/train_tribunals.json


In [ ]:
import ijson

path = acts_path

with open(path, "rb") as f:
    first = next(ijson.items(f, "item"))

print(type(first))
print(first.keys() if isinstance(first, dict) else first)

<class 'dict'>
dict_keys(['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars'])


In [ ]:
import ijson
import random
import json
import os


def reservoir_sample_json(
    path,
    n_samples,
    seed=42,
):
    rng = random.Random(seed)

    reservoir = []
    seen = 0

    with open(path, "rb") as f:

        for record in ijson.items(f, "item"):

            if not isinstance(record, dict):
                continue

            text = record.get("text")

            if not isinstance(text, str):
                continue

            seen += 1

            if len(reservoir) < n_samples:
                reservoir.append(text)

            else:
                j = rng.randint(0, seen - 1)

                if j < n_samples:
                    reservoir[j] = text

    print(
        f"{os.path.basename(path)}: "
        f"scanned {seen:,} documents, "
        f"selected {len(reservoir):,}"
    )

    return reservoir

In [ ]:
acts = reservoir_sample_json(
    acts_path,
    n_samples=25_000,
    seed=42,
)

high_courts = reservoir_sample_json(
    high_courts_path,
    n_samples=250_000,
    seed=43,
)

supreme_court = reservoir_sample_json(
    supreme_court_path,
    n_samples=25_000,
    seed=44,
)

tribunals = reservoir_sample_json(
    tribunals_path,
    n_samples=80_000,
    seed=45,
)

train_acts.json: scanned 18,826 documents, selected 18,826
train_high_courts.json: scanned 2,536,738 documents, selected 250,000
train_supreme_court.json: scanned 55,549 documents, selected 25,000
train_tribunals.json: scanned 869,132 documents, selected 80,000


In [ ]:
rng = random.Random(42)

sampled_texts = (
    acts
    + high_courts
    + supreme_court
    + tribunals
)

rng.shuffle(sampled_texts)

print(f"Total documents: {len(sampled_texts):,}")

Total documents: 373,826


In [ ]:
import os
from google.colab import drive
from datasets import load_from_disk, Dataset, DatasetDict

# Mount Google Drive
drive.mount('/content/drive')

# Define the Drive path as requested
cache_dir = '/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache'
os.makedirs(cache_dir, exist_ok=True)


iterable_dataset = load_legal_dataset(
    hf_token=hf_token,
    phase="phase_1",
    streaming=True
)


import os
import psutil
from datasets import IterableDataset

process = psutil.Process(os.getpid())


Mounted at /content/drive


README.md:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

In [ ]:
sample_path = os.path.join(
    cache_dir,
    "bpe_training_sample.json",
)

with open(sample_path, "w", encoding="utf-8") as f:

    for text in sampled_texts:

        f.write(
            json.dumps(
                {"text": text},
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"Saved to: {sample_path}")


In [ ]:
def local_text_stream(path):

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            record = json.loads(line)

            text = record.get("text")

            if isinstance(text, str):
                yield text

In [ ]:
import os
import re
import json
import heapq
import collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:                       # psutil optional
    _PROC = None


def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0


# ============================================================
# MUST BE IDENTICAL FOR TRAINING AND INFERENCE
# ============================================================
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


# ---- multiprocessing helpers (module level so they can be pickled) ----
_WORKER = None

def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:

    _HEAP_THRESHOLD = 24        # words longer than this use the heap merger
    _CACHE_LIMIT = 500_000

    # ========================================================
    # INIT
    # ========================================================
    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = (
            reserved_tokens
            if reserved_tokens is not None
            else ["<pad>", "<unk>", "<s>", "</s>"]
        )

        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}

        # (id_a, id_b) -> merge rank. Rank r created token first_merge_id + r.
        self.merges = {}

        self._ready = False
        self._cache = {}

    # ========================================================
    # SAVE / LOAD
    # ========================================================
    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "end_marker": self.end_marker,
            "reserved_tokens": self.reserved_tokens,
            "itos": self.itos,
            "stoi": self.stoi,
            "merges": [[list(p), r] for p, r in self.merges.items()],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)   # no indent: much smaller/faster

        print(f"Tokenizer saved to: {path}")
        print(f"Vocabulary size: {len(self.itos):,}")
        print(f"Number of merges: {len(self.merges):,}")

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get(
                "reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]
            ),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    # ========================================================
    # BASE VOCAB
    # ========================================================
    def _initialize_bytes(self):
        """Reserved tokens, then 256 byte tokens, then </w>."""
        for b in range(256):
            tok = bytes([b]).decode("latin-1")
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

        if self.end_marker not in self.stoi:
            self.stoi[self.end_marker] = len(self.itos)
            self.itos.append(self.end_marker)

    # ========================================================
    # WORD FREQUENCIES  (memory-safe)
    # ========================================================
    def _get_initial_vocab(
        self,
        corpus_texts,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        """
        Count words as plain strings (far lighter than tuples of ints),
        periodically dropping the long tail (typos, ids, citation numbers),
        with a hard cap on the number of unique words. Only the survivors are
        converted to byte-id tuples.

        Dropped rare words are still encodable at inference through byte
        fallback, so this costs almost nothing in tokenizer quality.
        """
        self._initialize_bytes()
        end_id = self.stoi[self.end_marker]

        counts = collections.Counter()
        threshold = prune_threshold
        n = 0
        last_ram = _ram_gb()

        def prune(th):
            return collections.Counter(
                {w: c for w, c in counts.items() if c >= th}
            )

        for n, text in enumerate(corpus_texts, 1):
            if not isinstance(text, str):
                continue

            if len(text) > max_doc_chars:
                print(
                    f"  [warn] example {n:,} has {len(text):,} chars "
                    f"-> truncated to {max_doc_chars:,}"
                )
                text = text[:max_doc_chars]

            # generator: never builds a giant list of words in memory
            counts.update(m.group() for m in WORD_RE.finditer(text))

            if n % 100 == 0:
                ram = _ram_gb()
                if ram - last_ram > 0.3:
                    print(f"  [warn] RAM jumped {last_ram:.2f} -> {ram:.2f} GB near example {n:,}")
                last_ram = ram

            if n % prune_every == 0:
                counts = prune(threshold)

            # hard safety cap: raise threshold until we fit
            while len(counts) > max_unique_words:
                threshold += 1
                counts = prune(threshold)
                print(f"  cap hit -> prune threshold raised to {threshold}")

            if n % 1000 == 0:
                print(
                    f"Examples: {n:,} | Unique words: {len(counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if n >= max_examples:
                break

        counts = prune(prune_threshold)     # final prune

        # convert survivors to id tuples
        byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        word_freqs = {}
        for w, c in counts.items():
            ids = tuple(byte_ids[b] for b in w.encode("utf-8")) + (end_id,)
            word_freqs[ids] = word_freqs.get(ids, 0) + c
        del counts

        print(f"\nFinished reading corpus. Examples: {n:,}")
        print(f"Unique words kept: {len(word_freqs):,}")
        return word_freqs

    # ========================================================
    # TRAIN
    # ========================================================
    def train(
        self,
        corpus_texts,
        save_checkpoints=None,
        checkpoint_dir=None,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        save_checkpoints = set(save_checkpoints or [])
        checkpoint_dir = checkpoint_dir or "."
        os.makedirs(checkpoint_dir, exist_ok=True)

        # ---- Step 1: word frequencies --------------------------------
        print("\nSTEP 1: word frequencies")
        word_freqs = self._get_initial_vocab(
            corpus_texts,
            max_examples=max_examples,
            prune_every=prune_every,
            prune_threshold=prune_threshold,
            max_unique_words=max_unique_words,
            max_doc_chars=max_doc_chars,
        )

        words = [list(w) for w in word_freqs]
        freqs = list(word_freqs.values())
        del word_freqs

        # ---- Step 2: pair statistics ---------------------------------
        print("\nSTEP 2: pair statistics")
        pair_counts = collections.defaultdict(int)
        pair_to_words = collections.defaultdict(set)     # pair -> word indices

        for idx, (w, f) in enumerate(zip(words, freqs)):
            for pair in zip(w, w[1:]):
                pair_counts[pair] += f
                pair_to_words[pair].add(idx)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        print(f"Initial vocab size: {len(self.itos):,}")
        print(f"Unique words:       {len(words):,}")
        print(f"Unique pairs:       {len(pair_counts):,}")
        print(f"RAM: {_ram_gb():.2f} GB")

        # ---- Step 3: merges ------------------------------------------
        print("\nSTEP 3: BPE merges")

        while len(self.itos) < self.vocab_size:

            # pop until an up-to-date entry is found (lazy deletion)
            best_pair, best_freq = None, 0
            while heap:
                neg, pair = heapq.heappop(heap)
                if -neg > 0 and pair_counts.get(pair, 0) == -neg:
                    best_pair, best_freq = pair, -neg
                    break

            if best_pair is None:
                print("No more pairs available.")
                break

            a, b = best_pair

            new_id = len(self.itos)
            new_str = self.itos[a] + self.itos[b]
            self.itos.append(new_str)
            self.stoi.setdefault(new_str, new_id)
            self.merges[best_pair] = len(self.merges)

            affected = pair_to_words.pop(best_pair, set())
            changed = set()

            for idx in affected:
                w, f = words[idx], freqs[idx]

                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                    if p != best_pair:
                        s = pair_to_words.get(p)
                        if s is not None:
                            s.discard(idx)

                out, i, n = [], 0, len(w)
                while i < n:
                    if i < n - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                    else:
                        out.append(w[i])
                        i += 1
                words[idx] = out

                for p in zip(out, out[1:]):
                    pair_counts[p] += f
                    pair_to_words[p].add(idx)
                    changed.add(p)

            pair_counts.pop(best_pair, None)
            changed.discard(best_pair)

            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_to_words.pop(p, None)

            # rebuild heap if it is mostly stale (bounds heap memory)
            if len(heap) > 4 * len(pair_counts) + 10_000:
                heap = [(-c, p) for p, c in pair_counts.items()]
                heapq.heapify(heap)

            vocab_now = len(self.itos)

            if vocab_now % 1000 == 0:
                print(
                    f"Vocab: {vocab_now:,} | merge freq: {best_freq:,} | "
                    f"affected: {len(affected):,} | pairs: {len(pair_counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if vocab_now in save_checkpoints:
                self.save(os.path.join(checkpoint_dir, f"bpe_{vocab_now}.json"))

        self._ready = False
        self._cache = {}

        print("\nBPE training complete.")
        print(f"Final vocabulary size: {len(self.itos):,}")
        print(f"Merges learned:        {len(self.merges):,}")

    # ========================================================
    # INFERENCE SETUP
    # ========================================================
    def _prepare_inference(self):
        if self._ready:
            return

        self._byte_ids = [
            self.stoi[bytes([b]).decode("latin-1")] for b in range(256)
        ]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]

        # sanity check: rank r must have created token first_merge_id + r
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError(
                    "Merge/vocab mismatch: cannot infer merge ids from ranks."
                )

        # integer keys: faster than tuple keys
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}

        self._cache = {}
        self._ready = True

    # ========================================================
    # MERGE STRATEGIES
    # ========================================================
    def _merge_small(self, ids):
        """Repeated best-rank scan. Fastest for short words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i

            if best is None:
                break

            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        return ids

    def _merge_heap(self, ids):
        """Linked list + min-heap: O(n log n) for long words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        n = len(ids)
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n

        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)

        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue                        # stale entry

            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i

            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))

        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit

        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)

        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)

        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    # ========================================================
    # PUBLIC API
    # ========================================================
    def encode(self, text, add_special_tokens=True):
        """Fast path: token ids only."""
        self._prepare_inference()

        out = [self._bos] if add_special_tokens else []
        extend = out.extend
        cache_get = self._cache.get
        bpe = self._bpe_word

        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None:
                t = bpe(w)
            extend(t)

        if add_special_tokens:
            out.append(self._eos)
        return out

    def tokenize(self, text, add_special_tokens=True):
        """Returns (token_ids, token_strings). Use encode() for bulk work."""
        ids = self.encode(text, add_special_tokens)
        itos = self.itos
        return ids, [itos[i] for i in ids]

    def encode_batch(self, texts, add_special_tokens=True,
                     num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)

        if num_workers is None:
            num_workers = os.cpu_count() or 1

        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]

        import multiprocessing as mp
        chunks = [
            (texts[i:i + chunk_size], add_special_tokens)
            for i in range(0, len(texts), chunk_size)
        ]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker,
                      initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        """Decodes a list of token IDs back into a string, handling special tokens."""
        self._prepare_inference()
        # Filter out special tokens for a cleaner decoded string
        filtered_ids = [
            tid for tid in token_ids
            if self.itos[tid] not in self.reserved_tokens
        ]
        # Join the token strings, replacing </w> with a space
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()


In [ ]:
# vocab_size = 65536
# tokenizer = BPETokenizer(vocab_size=vocab_size)

# print(f"Training BPE tokenizer with vocab size {vocab_size}...")

# tokenizer = BPETokenizer(
#     vocab_size=vocab_size,
#     reserved_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>", "<sep>", "<cls>"],
# )
# tokenizer.train(
#     local_text_stream(sample_path),
#     save_checkpoints=[8192, 16384, 32768, 65536],
#     checkpoint_dir=cache_dir,
#     max_examples=25_000,        # more coverage, see the caveat below
#     max_doc_chars=200_000,
#     prune_every=5_000,
#     prune_threshold=2,          # keeps mid-frequency words
#     max_unique_words=2_000_000, # about 300 MB, protects RAM
# )

# print("Tokenizer training finished!")

# bpe_en_path = os.path.join(cache_dir, 'final_bpe_en.json')
# tokenizer.save(bpe_en_path)


Training BPE tokenizer with vocab size 65536...

STEP 1: word frequencies
Examples: 1,000 | Unique words: 38,243 | RAM: 34.74 GB
  [warn] example 1,101 has 224,780 chars -> truncated to 200,000
  [warn] example 1,117 has 212,349 chars -> truncated to 200,000
  [warn] example 1,413 has 204,145 chars -> truncated to 200,000
  [warn] example 1,731 has 233,584 chars -> truncated to 200,000
Examples: 2,000 | Unique words: 56,280 | RAM: 34.74 GB
  [warn] example 2,043 has 233,216 chars -> truncated to 200,000
Examples: 3,000 | Unique words: 71,050 | RAM: 34.74 GB
  [warn] example 3,568 has 672,618 chars -> truncated to 200,000
  [warn] example 3,716 has 200,471 chars -> truncated to 200,000
  [warn] example 3,808 has 213,103 chars -> truncated to 200,000
Examples: 4,000 | Unique words: 84,897 | RAM: 34.74 GB
  [warn] example 4,819 has 241,619 chars -> truncated to 200,000
  [warn] example 4,957 has 318,723 chars -> truncated to 200,000
Examples: 5,000 | Unique words: 53,411 | RAM: 34.74 GB
 

In [ ]:
# ============================================================
# BPE TOKENIZER TEST
# ============================================================
_VOCAB_SIZE = 16384
bpe_en_path = os.path.join(cache_dir, f'bpe_{_VOCAB_SIZE}.json')
tokenizer = BPETokenizer.from_json(bpe_en_path)

print("Tokenizer loaded successfully")
print(f"Vocabulary size: {len(tokenizer.itos):,}")
print(f"Number of merges: {len(tokenizer.merges):,}")

test_texts = [
    "The Supreme Court of India held that the right to equality is a fundamental right.",

    "Section 302 of the Indian Penal Code provides punishment for murder.",

    "The petitioner challenged the constitutional validity of the impugned provision.",

    "Section 21: Protection of life and personal liberty shall not be denied.",
]

for text in test_texts:

    token_ids, token_strings = tokenizer.tokenize(
        text,
        add_special_tokens=True,
    )

    print("\n" + "=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKEN IDS:")
    print(token_ids)

    print("\nTOKENS:")
    print(token_strings)

    print("\nSTATISTICS:")
    print(f"Characters : {len(text):,}")
    print(f"Tokens     : {len(token_ids):,}")
    print(
        f"Chars/token: "
        f"{len(text) / max(len(token_ids), 1):.2f}"
    )

Tokenizer loaded successfully
Vocabulary size: 16,384
Number of merges: 16,120

TEXT:
The Supreme Court of India held that the right to equality is a fundamental right.

TOKEN IDS:
[2, 362, 1830, 464, 276, 827, 910, 330, 271, 1339, 291, 8717, 302, 300, 6254, 1339, 267, 3]

TOKENS:
['<s>', 'The</w>', 'Supreme</w>', 'Court</w>', 'of</w>', 'India</w>', 'held</w>', 'that</w>', 'the</w>', 'right</w>', 'to</w>', 'equality</w>', 'is</w>', 'a</w>', 'fundamental</w>', 'right</w>', '.</w>', '</s>']

STATISTICS:
Characters : 82
Tokens     : 18
Chars/token: 4.56

TEXT:
Section 302 of the Indian Penal Code provides punishment for murder.

TOKEN IDS:
[2, 589, 4348, 276, 271, 1835, 3448, 1523, 3210, 3434, 328, 5258, 267, 3]

TOKENS:
['<s>', 'Section</w>', '302</w>', 'of</w>', 'the</w>', 'Indian</w>', 'Penal</w>', 'Code</w>', 'provides</w>', 'punishment</w>', 'for</w>', 'murder</w>', '.</w>', '</s>']

STATISTICS:
Characters : 68
Tokens     : 14
Chars/token: 4.86

TEXT:
The petitioner challenged the co

In [ ]:
text = "reiterated by this Court observing that acquisition proceedings should be challenged before the same attain finality , in State of Mysore Vs . V . K . Kangan AIR 1975 SC 2190 ; [ PT . Girdharan Prasad Missir Vs . State of Bihar ] ( 1980 ) 2 SCC 83 ; Bhoop Singh Vs . Union of India AIR 1992 SC 1414 ; State of Orissa Vs . Dhobei Sethi & Anr . ( 1995 ) 5 SCC 583 ; State of Maharashtra Vs . Digambar AIR 1995 SC 1991 ; State of Tamil Nadu Vs . L . Krishnan AIR 1996 SC 497 ; and [ C . Padma & Ors . Vs . Dy . Secretary to Govt . of Tamil Nadu & Ors . ] ( 1997 ) 2 SCC 627 . 9 . In [ Municipal Corporation of Greater Bombay Vs . Industrial Development Investment Co . Pvt . Ltd . & Ors . ] AIR 1997 SC 482 , this Court observed as under : - \" If the interested person allows the grass to grow under his feet by allowing the acquisition proceedings to go on and reach its terminus in the award and possession is taken in furtherance thereof and vest in the State free from all encumbrances , the slumbered interested person would be told off the gates of the Court"

ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print("ORIGINAL:")
print(text)

print("\nIDS:")
print(ids)

print("\nDECODED:")
print(decoded)

print("\nMATCH:")
print(text == decoded)

ORIGINAL:
reiterated by this Court observing that acquisition proceedings should be challenged before the same attain finality , in State of Mysore Vs . V . K . Kangan AIR 1975 SC 2190 ; [ PT . Girdharan Prasad Missir Vs . State of Bihar ] ( 1980 ) 2 SCC 83 ; Bhoop Singh Vs . Union of India AIR 1992 SC 1414 ; State of Orissa Vs . Dhobei Sethi & Anr . ( 1995 ) 5 SCC 583 ; State of Maharashtra Vs . Digambar AIR 1995 SC 1991 ; State of Tamil Nadu Vs . L . Krishnan AIR 1996 SC 497 ; and [ C . Padma & Ors . Vs . Dy . Secretary to Govt . of Tamil Nadu & Ors . ] ( 1997 ) 2 SCC 627 . 9 . In [ Municipal Corporation of Greater Bombay Vs . Industrial Development Investment Co . Pvt . Ltd . & Ors . ] AIR 1997 SC 482 , this Court observed as under : - " If the interested person allows the grass to grow under his feet by allowing the acquisition proceedings to go on and reach its terminus in the award and possession is taken in furtherance thereof and vest in the State free from all encumbrances , t

In [ ]:
# ============================================================
# BPE TOKENIZER TEST
# ============================================================

_VOCAB_SIZE = 32768
bpe_en_path = os.path.join(cache_dir, f'bpe_{_VOCAB_SIZE}.json')
tokenizer = BPETokenizer.from_json(bpe_en_path)

print("Tokenizer loaded successfully")
print(f"Vocabulary size: {len(tokenizer.itos):,}")
print(f"Number of merges: {len(tokenizer.merges):,}")

test_texts = [
    "The Supreme Court of India held that the right to equality is a fundamental right.",

    "Section 302 of the Indian Penal Code provides punishment for murder.",

    "The petitioner challenged the constitutional validity of the impugned provision.",

    "Section 21: Protection of life and personal liberty shall not be denied.",
]

for text in test_texts:

    token_ids, token_strings = tokenizer.tokenize(
        text,
        add_special_tokens=True,
    )

    print("\n" + "=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKEN IDS:")
    print(token_ids)

    print("\nTOKENS:")
    print(token_strings)

    print("\nSTATISTICS:")
    print(f"Characters : {len(text):,}")
    print(f"Tokens     : {len(token_ids):,}")
    print(
        f"Chars/token: "
        f"{len(text) / max(len(token_ids), 1):.2f}"
    )

Tokenizer loaded successfully
Vocabulary size: 32,768
Number of merges: 32,504

TEXT:
The Supreme Court of India held that the right to equality is a fundamental right.

TOKEN IDS:
[2, 362, 1830, 464, 276, 827, 910, 330, 271, 1339, 291, 8717, 302, 300, 6254, 1339, 267, 3]

TOKENS:
['<s>', 'The</w>', 'Supreme</w>', 'Court</w>', 'of</w>', 'India</w>', 'held</w>', 'that</w>', 'the</w>', 'right</w>', 'to</w>', 'equality</w>', 'is</w>', 'a</w>', 'fundamental</w>', 'right</w>', '.</w>', '</s>']

STATISTICS:
Characters : 82
Tokens     : 18
Chars/token: 4.56

TEXT:
Section 302 of the Indian Penal Code provides punishment for murder.

TOKEN IDS:
[2, 589, 4348, 276, 271, 1835, 3448, 1523, 3210, 3434, 328, 5258, 267, 3]

TOKENS:
['<s>', 'Section</w>', '302</w>', 'of</w>', 'the</w>', 'Indian</w>', 'Penal</w>', 'Code</w>', 'provides</w>', 'punishment</w>', 'for</w>', 'murder</w>', '.</w>', '</s>']

STATISTICS:
Characters : 68
Tokens     : 14
Chars/token: 4.86

TEXT:
The petitioner challenged the co

In [ ]:
acts = reservoir_sample_json(
    acts_path,
    n_samples=20000,
    seed=42,
)

high_courts = reservoir_sample_json(
    high_courts_path,
    n_samples=300_000,
    seed=42,
)

supreme_court = reservoir_sample_json(
    supreme_court_path,
    n_samples=50_000,
    seed=42,
)

tribunals = reservoir_sample_json(
    tribunals_path,
    n_samples=250_000,
    seed=42,
)

train_acts.json: scanned 18,826 documents, selected 18,826
train_high_courts.json: scanned 2,536,738 documents, selected 300,000
train_supreme_court.json: scanned 55,549 documents, selected 50,000
train_tribunals.json: scanned 869,132 documents, selected 250,000


In [ ]:
# rng = random.Random(42)

train_val_texts = (
    acts
    + high_courts
    + supreme_court
    + tribunals
)

# rng.shuffle(train_val_texts)

print(f"Total documents: {len(train_val_texts):,}")

Total documents: 618,826


In [ ]:
# ============================================================
# DATA CLEANING
#   1. Unicode + whitespace normalisation
#   2. Language / script filtering (keep English, drop the rest)
#   3. De-duplication (exact + near-duplicate via MinHash-LSH)
#
# Run this AFTER the raw documents are sampled (train_val_texts)
# and BEFORE the train / val split, so that duplicates cannot
# leak from train into validation.
# ============================================================
import re
import zlib
import hashlib
import unicodedata
import collections
import numpy as np


# ------------------------------------------------------------
# 1. TEXT NORMALISATION
#    normalize_text() is also applied to the test prompts at
#    inference time so train and test text look the same.
# ------------------------------------------------------------
# characters that are deleted outright (zero-width, bidi marks, soft hyphen,
# BOM, the U+FFFD "decode error" marker)
_CHAR_MAP = {ord(c): None for c in (
    "\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u00ad\ufffd"
    "\u202a\u202b\u202c\u202d\u202e"
)}
# typographic quotes / dashes -> ASCII, so one symbol is not split over 3 variants
_CHAR_MAP.update({ord(c): "'" for c in "\u2018\u2019\u201a\u201b\u2032\u02bc"})
_CHAR_MAP.update({ord(c): '"' for c in "\u201c\u201d\u201e\u201f\u2033"})
_CHAR_MAP.update({ord(c): "-" for c in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"})

_CTRL_RE = re.compile(r"[\x00-\x08\x0e-\x1b\x7f\x80-\x84\x86-\x9f]")
_WS_RE = re.compile(r"\s+")                      # any run of unicode whitespace
_REPEAT_RE = re.compile(r"([^\w\s]|_)\1{3,}")    # '......', '-----', '_____' ...


def normalize_text(text, collapse_repeats=True):
    """Unicode-normalise and collapse whitespace.

    - quotes/dashes -> ASCII, NFKC (turns nbsp, thin/ideographic spaces,
      ligatures, full-width forms into plain characters)
    - removes zero-width / control / replacement characters
    - every run of whitespace (spaces, tabs, newlines, nbsp ...) -> one ' '
    - runs of 4+ identical punctuation chars ('.....', '-----') -> 3
    """
    text = text.translate(_CHAR_MAP)
    if not text.isascii():
        text = unicodedata.normalize("NFKC", text)
    text = _CTRL_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    if collapse_repeats:
        text = _REPEAT_RE.sub(lambda m: m.group(1) * 3, text)
    return text


# ------------------------------------------------------------
# 2. LANGUAGE / SCRIPT HANDLING (no external dependency)
# ------------------------------------------------------------
_LATIN = "A-Za-z\u00C0-\u024F\u1E00-\u1EFF"
_SCRIPTS = {
    "devanagari": "\u0900-\u097F", "bengali": "\u0980-\u09FF",
    "gurmukhi":   "\u0A00-\u0A7F", "gujarati": "\u0A80-\u0AFF",
    "odia":       "\u0B00-\u0B7F", "tamil":    "\u0B80-\u0BFF",
    "telugu":     "\u0C00-\u0C7F", "kannada":  "\u0C80-\u0CFF",
    "malayalam":  "\u0D00-\u0D7F",
    "arabic/urdu": "\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF",
    "cyrillic":   "\u0400-\u04FF",
    "cjk":        "\u3040-\u30FF\u3400-\u4DBF\u4E00-\u9FFF\uAC00-\uD7AF",
}
_SCRIPT_RES = {k: re.compile(f"[{v}]") for k, v in _SCRIPTS.items()}
_ALPHA_RE = re.compile(r"[^\W\d_]")
_LATIN_RE = re.compile(f"[{_LATIN}]")
# a run of non-Latin letters (plus the combining marks / joiners that belong to them)
_FOREIGN_RUN_RE = re.compile(
    "[" + "".join(_SCRIPTS.values()) + "]+"
    "(?:[\\s\u200c\u200d]*[" + "".join(_SCRIPTS.values()) + "]+)*"
)
_EN_WORD_RE = re.compile(r"[a-z]+")
_ENGLISH_STOP = frozenset("""
the of and to in that is was for by on with as be it at this which are from or not has have
had been an a he she they his her their its were any such under also but if all no can shall
may would
""".split())


def language_profile(text, sample_chars=3000):
    """Cheap profile from the start and middle of a document.
    Returns (latin_ratio, english_stopword_ratio, dominant_foreign_script)."""
    n = len(text)
    s = text if n <= 2 * sample_chars else (
        text[:sample_chars] + " " + text[n // 2: n // 2 + sample_chars])
    n_alpha = len(_ALPHA_RE.findall(s))
    if n_alpha == 0:
        return 0.0, 0.0, "none"
    n_latin = len(_LATIN_RE.findall(s))
    latin_ratio = n_latin / n_alpha

    words = _EN_WORD_RE.findall(s.lower())
    en_ratio = (sum(w in _ENGLISH_STOP for w in words) / len(words)) if words else 0.0

    foreign = "latin-non-english"
    if latin_ratio < 0.5:
        counts = {k: len(r.findall(s)) for k, r in _SCRIPT_RES.items()}
        best = max(counts, key=counts.get)
        foreign = best if counts[best] > 0 else "other"
    return latin_ratio, en_ratio, foreign


# ------------------------------------------------------------
# 3. NEAR-DUPLICATE DETECTION (MinHash + LSH banding)
# ------------------------------------------------------------
_WORD_RE = re.compile(r"\w+")
_MH_PRIME = np.uint64(4294967311)           # prime just above 2**32
_ROLL = np.uint64(1000003)


class NearDupIndex:
    """Streaming near-duplicate detector. The first copy seen is kept."""

    def __init__(self, num_perm=64, bands=8, threshold=0.8,
                 shingle=5, max_words=2000, seed=1):
        assert num_perm % bands == 0
        self.num_perm, self.bands = num_perm, bands
        self.rows = num_perm // bands
        self.threshold, self.shingle, self.max_words = threshold, shingle, max_words
        rng = np.random.RandomState(seed)
        self.a = rng.randint(1, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.b = rng.randint(0, 2 ** 31 - 1, size=num_perm).astype(np.uint64)[:, None]
        self.sigs = []
        self.buckets = [dict() for _ in range(bands)]

    def _shingle_hashes(self, words):
        words = words[: self.max_words]
        k = self.shingle
        wh = np.fromiter((zlib.crc32(w.encode()) for w in words),
                         dtype=np.uint64, count=len(words))
        m = len(wh) - k + 1
        if m < 1:
            return wh
        sh = wh[:m].copy()
        for j in range(1, k):                 # rolling hash of k consecutive words
            sh = sh * _ROLL + wh[j: m + j]
        return (sh ^ (sh >> np.uint64(32))) & np.uint64(0xFFFFFFFF)

    def check_and_add(self, words):
        """True if a near-duplicate was already indexed; otherwise index and return False."""
        h = self._shingle_hashes(words)[None, :]
        sig = ((self.a * h + self.b) % _MH_PRIME).min(axis=1)
        r = self.rows
        keys = [hash(sig[i * r:(i + 1) * r].tobytes()) for i in range(self.bands)]
        for i, k in enumerate(keys):
            j = self.buckets[i].get(k)
            if j is not None and (self.sigs[j] == sig).mean() >= self.threshold:
                return True
        idx = len(self.sigs)
        self.sigs.append(sig)
        for i, k in enumerate(keys):
            self.buckets[i].setdefault(k, idx)
        return False


# ------------------------------------------------------------
# PIPELINE
# ------------------------------------------------------------
def clean_corpus(
    texts,
    min_words=30,                 # drop near-empty documents
    language_filter=True,         # drop non-English documents
    min_latin_ratio=0.90,         # share of letters that must be Latin script
    min_english_ratio=0.12,       # share of words that are common English words
    strip_foreign_runs=True,      # remove stray Hindi/Urdu/... fragments inside English docs
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,          # estimated Jaccard similarity above which = duplicate
    collapse_repeats=True,
    n_examples=3,
    progress_every=20_000,
):
    stats = collections.Counter()
    dropped_scripts = collections.Counter()
    examples = collections.defaultdict(list)
    seen_exact = set()
    near = NearDupIndex(threshold=near_threshold) if near_dedup else None
    kept = []

    def drop(reason, text):
        stats[reason] += 1
        if len(examples[reason]) < n_examples:
            examples[reason].append(text[:200])

    for n, raw in enumerate(texts, 1):
        stats["input_docs"] += 1
        if not isinstance(raw, str):
            drop("not_a_string", str(raw)); continue

        # 1. normalise -------------------------------------------------
        text = normalize_text(raw, collapse_repeats=collapse_repeats)
        if text != raw:
            stats["docs_changed_by_normalisation"] += 1

        # 2. cheap length pre-check (so tiny docs are reported as "too short",
        #    not mislabelled as non-English)
        if text.count(" ") + 1 < min_words:
            drop("dropped_too_short", text); continue

        # 3. language --------------------------------------------------
        if language_filter:
            latin, en, script = language_profile(text)
            if latin < min_latin_ratio:
                dropped_scripts[script] += 1
                drop("dropped_non_latin_script", text); continue
            if en < min_english_ratio:
                dropped_scripts["latin-non-english"] += 1
                drop("dropped_non_english_latin", text); continue
            if strip_foreign_runs:
                cleaned = _FOREIGN_RUN_RE.sub(" ", text)
                if cleaned != text:
                    stats["docs_with_foreign_fragments_stripped"] += 1
                    text = _WS_RE.sub(" ", cleaned).strip()

        # 4. final length check (after foreign fragments were stripped) -
        words = _WORD_RE.findall(text.lower())
        if len(words) < min_words:
            drop("dropped_too_short", text); continue

        # 5. exact duplicates ------------------------------------------
        if exact_dedup:
            key = hashlib.blake2b(" ".join(words).encode(), digest_size=16).digest()
            if key in seen_exact:
                drop("dropped_exact_duplicate", text); continue
            seen_exact.add(key)

        # 6. near duplicates -------------------------------------------
        if near is not None and near.check_and_add(words):
            drop("dropped_near_duplicate", text); continue

        kept.append(text)
        if progress_every and n % progress_every == 0:
            print(f"  processed {n:,} | kept {len(kept):,}")

    stats["kept_docs"] = len(kept)
    return kept, {"stats": stats, "dropped_scripts": dropped_scripts, "examples": examples}


def print_clean_report(report, chars_before=None, chars_after=None):
    s = report["stats"]
    n = max(s["input_docs"], 1)
    print("=" * 62)
    print(f"Input documents                : {s['input_docs']:>9,}")
    for k in ["not_a_string", "dropped_non_latin_script", "dropped_non_english_latin",
              "dropped_too_short", "dropped_exact_duplicate", "dropped_near_duplicate"]:
        if s[k]:
            print(f"  - {k:<28}: {s[k]:>9,}  ({100 * s[k] / n:5.1f}%)")
    print(f"Kept documents                 : {s['kept_docs']:>9,}  ({100 * s['kept_docs'] / n:5.1f}%)")
    print(f"Docs changed by normalisation  : {s['docs_changed_by_normalisation']:>9,}")
    print(f"Docs with foreign text stripped: {s['docs_with_foreign_fragments_stripped']:>9,}")
    if chars_before and chars_after:
        print(f"Characters                     : {chars_before:,} -> {chars_after:,}")
    if report["dropped_scripts"]:
        print("\nLanguages / scripts removed:")
        for k, v in report["dropped_scripts"].most_common():
            print(f"  {k:<20}{v:>9,}")
    for reason, ex in report["examples"].items():
        print(f"\n[{reason}] sample:")
        for e in ex:
            print("   ", repr(e[:140]))
    print("=" * 62)


In [ ]:
chars_before = sum(len(t) for t in train_val_texts)

train_val_texts, clean_report = clean_corpus(
    train_val_texts,
    min_words=30,
    language_filter=True,
    min_latin_ratio=0.90,
    min_english_ratio=0.12,
    strip_foreign_runs=True,
    exact_dedup=True,
    near_dedup=True,
    near_threshold=0.80,
)

chars_after = sum(len(t) for t in train_val_texts)
print_clean_report(clean_report, chars_before, chars_after)

# free the raw per-source lists (they still hold the uncleaned text in RAM)
import gc
del acts, high_courts, supreme_court, tribunals
gc.collect()

rng = random.Random(42)
# rng.shuffle(train_val_texts)
print(f"Total documents after cleaning: {len(train_val_texts):,}")

  processed 20,000 | kept 19,035
  processed 40,000 | kept 38,546
  processed 60,000 | kept 57,767
  processed 80,000 | kept 76,848
  processed 100,000 | kept 95,793
  processed 120,000 | kept 114,672
  processed 140,000 | kept 133,464
  processed 160,000 | kept 152,248
  processed 180,000 | kept 170,886
  processed 200,000 | kept 189,472
  processed 220,000 | kept 208,028
  processed 240,000 | kept 226,656
  processed 260,000 | kept 245,267
  processed 280,000 | kept 263,838
  processed 300,000 | kept 282,247
  processed 320,000 | kept 300,726
  processed 340,000 | kept 320,447
  processed 360,000 | kept 340,120
  processed 380,000 | kept 358,892
  processed 400,000 | kept 376,988
  processed 420,000 | kept 394,885
  processed 460,000 | kept 430,164
  processed 480,000 | kept 447,606
  processed 500,000 | kept 465,182
  processed 520,000 | kept 482,668
  processed 540,000 | kept 500,396
  processed 560,000 | kept 518,116
  processed 580,000 | kept 535,857
  processed 600,000 | kept 55

In [ ]:
vocab_size = 32768
tokenizer = BPETokenizer(vocab_size=vocab_size)

print(f"Training BPE tokenizer with vocab size {vocab_size}...")

tokenizer = BPETokenizer(
    vocab_size=vocab_size,
    reserved_tokens=["<pad>", "<unk>", "<s>", "</s>", "<mask>", "<sep>", "<cls>"],
)
tokenizer.train(
    train_val_texts,
    save_checkpoints=[8192, 16384, 32768],
    checkpoint_dir=cache_dir
)

print("Tokenizer training finished!")

bpe_en_path = os.path.join(cache_dir, 'final_bpe_en.json')
tokenizer.save(bpe_en_path)


In [ ]:
# ============================================================
# BPE TOKENIZER TEST
# ============================================================

_VOCAB_SIZE = 32768
bpe_en_path = os.path.join(cache_dir, f'final_bpe_en.json')
tokenizer = BPETokenizer.from_json(bpe_en_path)

print("Tokenizer loaded successfully")
print(f"Vocabulary size: {len(tokenizer.itos):,}")
print(f"Number of merges: {len(tokenizer.merges):,}")

test_texts = [
    "The Supreme Court of India held that the right to equality is a fundamental right.",

    "Section 302 of the Indian Penal Code provides punishment for murder.",

    "The petitioner challenged the constitutional validity of the impugned provision.",

    "Section 21: Protection of life and personal liberty shall not be denied.",
]

for text in test_texts:

    token_ids, token_strings = tokenizer.tokenize(
        text,
        add_special_tokens=True,
    )

    print("\n" + "=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKEN IDS:")
    print(token_ids)

    print("\nTOKENS:")
    print(token_strings)

    print("\nSTATISTICS:")
    print(f"Characters : {len(text):,}")
    print(f"Tokens     : {len(token_ids):,}")
    print(
        f"Chars/token: "
        f"{len(text) / max(len(token_ids), 1):.2f}"
    )

Tokenizer loaded successfully
Vocabulary size: 32,768
Number of merges: 32,504

TEXT:
The Supreme Court of India held that the right to equality is a fundamental right.

TOKEN IDS:
[2, 371, 2154, 498, 278, 957, 1185, 351, 271, 1516, 293, 9656, 306, 304, 9169, 1516, 272, 3]

TOKENS:
['<s>', 'The</w>', 'Supreme</w>', 'Court</w>', 'of</w>', 'India</w>', 'held</w>', 'that</w>', 'the</w>', 'right</w>', 'to</w>', 'equality</w>', 'is</w>', 'a</w>', 'fundamental</w>', 'right</w>', '.</w>', '</s>']

STATISTICS:
Characters : 82
Tokens     : 18
Chars/token: 4.56

TEXT:
Section 302 of the Indian Penal Code provides punishment for murder.

TOKEN IDS:
[2, 594, 4658, 278, 271, 1596, 2896, 1342, 4689, 3364, 319, 7103, 272, 3]

TOKENS:
['<s>', 'Section</w>', '302</w>', 'of</w>', 'the</w>', 'Indian</w>', 'Penal</w>', 'Code</w>', 'provides</w>', 'punishment</w>', 'for</w>', 'murder</w>', '.</w>', '</s>']

STATISTICS:
Characters : 68
Tokens     : 14
Chars/token: 4.86

TEXT:
The petitioner challenged the c

In [ ]:
import random

rng = random.Random(42)

indices = list(range(len(train_val_texts)))
# rng.shuffle(indices)

val_size = int(
    0.05 * len(indices)
)

val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_texts = [
    train_val_texts[i]
    for i in train_indices
]

val_texts = [
    train_val_texts[i]
    for i in val_indices
]

print("Train:", len(train_texts))
print("Val:", len(val_texts))

Train: 541610
Val: 28505


In [ ]:
import gc
import numpy as np

def memory_safe_encode_and_save(texts, tokenizer, save_path, chunk_size=50000):
    """
    Encodes text dataset in small chunks and saves directly to disk to prevent RAM crashes.
    """
    print(f"Starting memory-safe encoding for {len(texts):,} documents...")

    # We will write the list of token IDs to a JSON-lines or partition file to keep RAM low
    with open(save_path, 'w', encoding='utf-8') as f:
        for i in range(0, len(texts), chunk_size):
            chunk = texts[i:i + chunk_size]
            print(f"  Processing chunk {i//chunk_size + 1} ({i:,} to {min(i+chunk_size, len(texts)):,})...")

            # Encode the small chunk
            encoded_chunk = tokenizer.encode_batch(
                chunk,
                add_special_tokens=True,
                num_workers=4,
                chunk_size=32
            )

            # Write to disk immediately
            for tokens in encoded_chunk:
                f.write(json.dumps(tokens) + '\n')

            # Free memory
            del chunk, encoded_chunk
            gc.collect()

    print(f"Finished encoding! Saved to {save_path}")

In [ ]:
# Define file paths for tokenized data on Google Drive
train_tokens_path = os.path.join(cache_dir, "train_tokens.jsonl")
val_tokens_path = os.path.join(cache_dir, "val_tokens.jsonl")

# Run memory-safe encoding
memory_safe_encode_and_save(train_texts, tokenizer, train_tokens_path)
memory_safe_encode_and_save(val_texts, tokenizer, val_tokens_path)

Starting memory-safe encoding for 541,610 documents...
  Processing chunk 1 (0 to 50,000)...
  Processing chunk 2 (50,000 to 100,000)...
  Processing chunk 3 (100,000 to 150,000)...
  Processing chunk 4 (150,000 to 200,000)...
  Processing chunk 5 (200,000 to 250,000)...
  Processing chunk 6 (250,000 to 300,000)...
  Processing chunk 7 (300,000 to 350,000)...
  Processing chunk 8 (350,000 to 400,000)...
  Processing chunk 9 (400,000 to 450,000)...
  Processing chunk 10 (450,000 to 500,000)...
  Processing chunk 11 (500,000 to 541,610)...
Finished encoding! Saved to /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/train_tokens.jsonl
Starting memory-safe encoding for 28,505 documents...
  Processing chunk 1 (0 to 28,505)...
Finished encoding! Saved to /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/val_tokens.jsonl


In [ ]:
# Load the tokens back from disk in a memory-light manner
def load_tokens_from_jsonl(path):
    tokens_list = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            tokens_list.append(json.loads(line))
    return tokens_list


# Define file paths for tokenized data on Google Drive
train_tokens_path = os.path.join(cache_dir, "train_tokens.jsonl")
val_tokens_path = os.path.join(cache_dir, "val_tokens.jsonl")

train_tokens = load_tokens_from_jsonl(train_tokens_path)
val_tokens = load_tokens_from_jsonl(val_tokens_path)
print(f"Loaded {len(train_tokens):,} train token sequences and {len(val_tokens):,} validation sequences.")

Loaded 541,610 train token sequences and 28,505 validation sequences.


In [ ]:
import torch
import os

def get_device():
    """Returns the available computing device (CUDA, MPS, or CPU)."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def save_checkpoint(model, optimizer, epoch, loss, filepath, best_loss=float('inf'), loss_history=None, global_step=0):
    """Saves the model and optimizer state to a checkpoint file."""
    os.makedirs(os.path.dirname(filepath) or '.', exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer else None,
        'loss': loss,
        'best_loss': best_loss,
        'loss_history': loss_history if loss_history is not None else [],
        'step': global_step, # Save global_step
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved to {filepath}")

def load_checkpoint(filepath, model, optimizer=None):
    """Loads a model and optimizer from a checkpoint file."""
    if not os.path.exists(filepath):
        print(f"No checkpoint found at {filepath}")
        return 0, float('inf'), float('inf'), [], 0 # Added global_step as 0

    checkpoint = torch.load(filepath, map_location=get_device())
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer and checkpoint.get('optimizer_state_dict'):
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint.get('epoch', 0)
    loss = checkpoint.get('loss', float('inf'))
    best_loss = checkpoint.get('best_loss', float('inf'))
    loss_history = checkpoint.get('loss_history', [])
    global_step = checkpoint.get('step', 0) # Load global_step

    print(f"Loaded checkpoint from '{filepath}' (Epoch {epoch}, Step {global_step})")

    return epoch, loss, best_loss, loss_history, global_step # Return global_step


In [ ]:
device = get_device()
print("Using device: ", device)

Using device:  cuda


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class GQLARotaryEmbedding(nn.Module):
    """
    Rotary embedding used only on the decoupled RoPE pathway.

    GQLA:
        q^R_t = RoPE_t(W_QR c^Q_t)
        k^R_t = RoPE_t(W_KR x_t)

    Input:
        x:
            [B, H, T, rope_dim]

        position_ids:
            [B, T]
    """

    def __init__(
        self,
        dim: int,
        base: float = 10000.0,
    ):
        super().__init__()

        if dim % 2 != 0:
            raise ValueError(
                f"RoPE dimension must be even, got {dim}"
            )

        self.dim = dim
        self.base = base

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    dim,
                    2,
                    dtype=torch.float32
                ) / dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

    @staticmethod
    def rotate_half(x):
        half = x.shape[-1] // 2

        x1 = x[..., :half]
        x2 = x[..., half:]

        return torch.cat(
            [-x2, x1],
            dim=-1
        )

    def forward(
        self,
        x,
        position_ids,
    ):
        """
        x:
            [B, H, T, dim]

        position_ids:
            [B, T]
        """

        # [B, T, dim/2]
        freqs = torch.einsum(
            "bt,d->btd",
            position_ids.float(),
            self.inv_freq
        )

        # [B, T, dim]
        emb = torch.cat(
            [freqs, freqs],
            dim=-1
        )

        cos = emb.cos().unsqueeze(1)
        sin = emb.sin().unsqueeze(1)

        x_rope = x[..., :self.dim]

        x_rope = (
            x_rope * cos
            + self.rotate_half(x_rope) * sin
        )

        return x_rope



In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# GQLA
# ============================================================

class GroupQueryLatentAttention(nn.Module):

    def __init__(
        self,
        d_model: int,
        num_query_heads: int,
        num_kv_groups: int,
        head_dim: int,
        kv_latent_dim: int,
        rope_dim: int,
        q_latent_dim: int = None,
        dropout: float = 0.1,
        rope_base: float = 10000.0,
        path: str = "gqa",
    ):
        super().__init__()

        if num_query_heads % num_kv_groups != 0:
            raise ValueError(
                "num_query_heads must be divisible by "
                "num_kv_groups"
            )

        if path not in {"gqa", "mqa_absorb"}:
            raise ValueError(
                "path must be either 'gqa' or 'mqa_absorb'"
            )

        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_groups = num_kv_groups
        self.head_dim = head_dim

        self.kv_latent_dim = kv_latent_dim
        self.rope_dim = rope_dim
        self.path = path # Add this line to store the path as an instance attribute

        self.heads_per_group = (
            num_query_heads // num_kv_groups
        )

        # ----------------------------------------------------
        # Q-side latent
        #
        # c_Q = W_DQ x
        # q_C = W_UQ c_Q
        # ----------------------------------------------------

        if q_latent_dim is None:
            # Paper explicitly notes that for models such as
            # LLaMA-3-8B and Qwen2.5-7B this factorization
            # collapses to W_Q directly.
            q_latent_dim = d_model

        self.q_latent_dim = q_latent_dim

        self.W_DQ = nn.Linear(
            d_model,
            q_latent_dim,
            bias=False
        )

        self.W_UQ = nn.Linear(
            q_latent_dim,
            num_query_heads * head_dim,
            bias=False
        )

        # ----------------------------------------------------
        # KV latent
        #
        # c_KV = W_DKV x
        # ----------------------------------------------------

        self.W_DKV = nn.Linear(
            d_model,
            kv_latent_dim,
            bias=False
        )

        # ----------------------------------------------------
        # GROUP indexed up-projections
        #
        # W_UK, W_UV:
        #
        # [g * d_h, r_KV]
        #
        # NOT:
        #
        # [h_q * d_h, r_KV]
        #
        # This group indexing is the key GQLA change.
        # ----------------------------------------------------

        self.W_UK = nn.Linear(
            kv_latent_dim,
            num_kv_groups * head_dim,
            bias=False
        )

        self.W_UV = nn.Linear(
            kv_latent_dim,
            num_kv_groups * head_dim,
            bias=False
        )

        # ----------------------------------------------------
        # Decoupled RoPE pathway
        #
        # q_R = RoPE(W_QR c_Q)
        # k_R = RoPE(W_KR x)
        # ----------------------------------------------------

        self.W_QR = nn.Linear(
            q_latent_dim,
            rope_dim,
            bias=False
        )

        self.W_KR = nn.Linear(
            d_model,
            rope_dim,
            bias=False
        )

        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        self.W_O = nn.Linear(
            num_query_heads * head_dim,
            d_model,
            bias=False
        )

        self.attn_dropout = nn.Dropout(dropout)

        self.rotary_emb = GQLARotaryEmbedding(
            dim=rope_dim,
            base=rope_base
        )

    # ========================================================
    # Helpers
    # ========================================================

    def _get_group_indices(self, device):
        """
        Maps each query head to its KV group.

        Example:

            hq = 8
            g  = 2

            query heads:

                0 1 2 3 | 4 5 6 7
                --------+--------
                  group 0 | group 1
        """

        return (
            torch.arange(
                self.num_query_heads,
                device=device
            )
            // self.heads_per_group
        )

    def _causal_mask(
        self,
        q_len,
        kv_len,
        past_len,
        device,
    ):
        """
        Correct causal mask when cached tokens exist.

        Example:

        past_len = 3
        q_len   = 2
        kv_len  = 5

        query position 3 can see:
            0 1 2 3

        query position 4 can see:
            0 1 2 3 4
        """

        return torch.triu(
            torch.ones(
                q_len,
                kv_len,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1 + past_len
        )

    # ========================================================
    # GQA decoding path
    # ========================================================

    def _forward_gqa(
        self,
        hidden_states,
        c_Q,
        c_KV,
        q_R,
        k_R_new,
        past_key_value,
        attention_mask,
        use_cache,
    ):
        """
        GQLA GQA path.

        Cache:

            K_C : [B, g, S, d_h]
            V_C : [B, g, S, d_h]
            K_R : [B, 1, S, d_R]

        This corresponds to the paper's per-group
        expanded GQA cache.
        """

        B, T, _ = hidden_states.shape

        # ----------------------------------------------------
        # Content query
        # ----------------------------------------------------

        q_C = self.W_UQ(c_Q)

        q_C = q_C.view(
            B,
            T,
            self.num_query_heads,
            self.head_dim
        )

        q_C = q_C.transpose(1, 2)

        # ----------------------------------------------------
        # Group K/V
        # ----------------------------------------------------

        k_C = self.W_UK(c_KV)

        k_C = k_C.view(
            B,
            T,
            self.num_kv_groups,
            self.head_dim
        )

        k_C = k_C.transpose(1, 2)

        v_C = self.W_UV(c_KV)

        v_C = v_C.view(
            B,
            T,
            self.num_kv_groups,
            self.head_dim
        )

        v_C = v_C.transpose(1, 2)

        # ----------------------------------------------------
        # Cache
        #
        # IMPORTANT:
        # We cache GROUP K/V, not repeated query-head K/V.
        # ----------------------------------------------------

        if past_key_value is not None:

            past_k_C, past_v_C, past_k_R = (
                past_key_value
            )

            past_len = past_k_C.shape[2]

            k_C = torch.cat(
                [past_k_C, k_C],
                dim=2
            )

            v_C = torch.cat(
                [past_v_C, v_C],
                dim=2
            )

            k_R = torch.cat(
                [past_k_R, k_R_new],
                dim=2
            )

        else:

            past_len = 0

            k_R = k_R_new

        kv_len = k_C.shape[2]

        # ----------------------------------------------------
        # Position IDs
        # ----------------------------------------------------

        q_positions = torch.arange(
            past_len,
            past_len + T,
            device=hidden_states.device
        ).unsqueeze(0).expand(B, -1)

        k_positions = torch.arange(
            kv_len,
            device=hidden_states.device
        ).unsqueeze(0).expand(B, -1)

        # ----------------------------------------------------
        # RoPE
        # ----------------------------------------------------

        q_R = self.rotary_emb(
            q_R,
            q_positions
        )

        k_R = self.rotary_emb(
            k_R,
            k_positions
        )

        # ----------------------------------------------------
        # Repeat group K/V across query heads
        #
        # g = 2
        # hq = 8
        #
        # group 0 -> query heads 0,1,2,3
        # group 1 -> query heads 4,5,6,7
        # ----------------------------------------------------

        k_C_attn = k_C.repeat_interleave(
            self.heads_per_group,
            dim=1
        )

        v_C_attn = v_C.repeat_interleave(
            self.heads_per_group,
            dim=1
        )

        k_R_attn = k_R.expand(
            -1,
            self.num_query_heads,
            -1,
            -1
        )

        q_R_attn = q_R.expand(
            -1,
            self.num_query_heads,
            -1,
            -1
        )

        # ----------------------------------------------------
        # Construct complete Q/K
        #
        # q_i = [q_C_i ; q_R_i]
        #
        # k_i = [k_C_group(i) ; k_R]
        # ----------------------------------------------------

        q = torch.cat(
            [q_C, q_R_attn],
            dim=-1
        )

        k = torch.cat(
            [k_C_attn, k_R_attn],
            dim=-1
        )

        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        scale = math.sqrt(
            self.head_dim + self.rope_dim
        )

        attn_scores = torch.matmul(
            q,
            k.transpose(-1, -2)
        ) / scale

        # ----------------------------------------------------
        # Causal mask
        # ----------------------------------------------------

        causal_mask = self._causal_mask(
            q_len=T,
            kv_len=kv_len,
            past_len=past_len,
            device=hidden_states.device
        )

        attn_scores = attn_scores.masked_fill(
            causal_mask.unsqueeze(0).unsqueeze(0),
            torch.finfo(attn_scores.dtype).min
        )

        # ----------------------------------------------------
        # External mask
        # ----------------------------------------------------

        if attention_mask is not None:

            attn_scores = (
                attn_scores + attention_mask
            )

        # ----------------------------------------------------
        # Softmax
        # ----------------------------------------------------

        attn_weights = F.softmax(
            attn_scores.float(),
            dim=-1
        ).to(attn_scores.dtype)

        attn_weights = self.attn_dropout(
            attn_weights
        )

        # ----------------------------------------------------
        # Values
        # ----------------------------------------------------

        attn_output = torch.matmul(
            attn_weights,
            v_C_attn
        )

        # [B,H,T,D] -> [B,T,H*D]

        attn_output = (
            attn_output
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                T,
                self.num_query_heads * self.head_dim
            )
        )

        output = self.W_O(
            attn_output
        )

        # ----------------------------------------------------
        # Cache only if explicitly requested
        #
        # DETACH is important.
        # ----------------------------------------------------

        if use_cache:

            new_cache = (
                k_C.detach(),
                v_C.detach(),
                k_R.detach()
            )

        else:

            new_cache = None

        return output, new_cache

    # ========================================================
    # MQA-absorb path
    # ========================================================

    def _forward_mqa_absorb(
        self,
        hidden_states,
        c_Q,
        c_KV,
        q_R,
        k_R_new,
        past_key_value,
        attention_mask,
        use_cache,
    ):
        """
        MLA-style MQA-absorb path.

        Cache:

            c_KV : [B, S, r_KV]
            k_R  : [B, S, d_R]

        This is the compact latent cache described in Eq. (3)
        of the paper.
        """

        B, T, _ = hidden_states.shape

        # ----------------------------------------------------
        # Content query
        # ----------------------------------------------------

        q_C = self.W_UQ(c_Q)

        q_C = q_C.view(
            B,
            T,
            self.num_query_heads,
            self.head_dim
        )

        q_C = q_C.transpose(1, 2)

        # ----------------------------------------------------
        # Cache latent
        # ----------------------------------------------------

        if past_key_value is not None:

            past_c_KV, past_k_R = (
                past_key_value
            )

            past_len = past_c_KV.shape[1]

            c_KV = torch.cat(
                [past_c_KV, c_KV],
                dim=1
            )

            k_R = torch.cat(
                [
                    past_k_R,
                    k_R_new.squeeze(1)
                ],
                dim=1
            )

        else:

            past_len = 0

            k_R = k_R_new.squeeze(1)

        kv_len = c_KV.shape[1]

        # ----------------------------------------------------
        # Positions
        # ----------------------------------------------------

        q_positions = torch.arange(
            past_len,
            past_len + T,
            device=hidden_states.device
        ).unsqueeze(0).expand(B, -1)

        k_positions = torch.arange(
            kv_len,
            device=hidden_states.device
        ).unsqueeze(0).expand(B, -1)

        # ----------------------------------------------------
        # RoPE
        # ----------------------------------------------------

        q_R = self.rotary_emb(
            q_R,
            q_positions
        )

        k_R = self.rotary_emb(
            k_R.unsqueeze(1),
            k_positions
        )

        # ----------------------------------------------------
        # Absorb W_UK into Q
        #
        # q'_C = W_UK^T q_C
        # ----------------------------------------------------

        group_indices = self._get_group_indices(
            hidden_states.device
        )

        W_UK = self.W_UK.weight

        W_UK = W_UK.view(
            self.num_kv_groups,
            self.head_dim,
            self.kv_latent_dim
        )

        W_UK_heads = W_UK[
            group_indices
        ]

        q_latent = torch.einsum(
            "bhtd,hdr->bhtr",
            q_C,
            W_UK_heads
        )

        # ----------------------------------------------------
        # q = [W_UK^T q_C ; q_R]
        # k = [c_KV ; k_R]
        # ----------------------------------------------------

        q = torch.cat(
            [
                q_latent,
                q_R.expand(
                    -1,
                    self.num_query_heads,
                    -1,
                    -1
                )
            ],
            dim=-1
        )

        k_latent = c_KV.unsqueeze(1).expand(
            -1,
            self.num_query_heads,
            -1,
            -1
        )

        k_rope = k_R.expand(
            -1,
            self.num_query_heads,
            -1,
            -1
        )

        k = torch.cat(
            [
                k_latent,
                k_rope
            ],
            dim=-1
        )

        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------

        scale = math.sqrt(
            self.kv_latent_dim + self.rope_dim
        )

        attn_scores = torch.matmul(
            q,
            k.transpose(-1, -2)
        ) / scale

        causal_mask = self._causal_mask(
            q_len=T,
            kv_len=kv_len,
            past_len=past_len,
            device=hidden_states.device
        )

        attn_scores = attn_scores.masked_fill(
            causal_mask.unsqueeze(0).unsqueeze(0),
            torch.finfo(attn_scores.dtype).min
        )

        if attention_mask is not None:

            attn_scores = (
                attn_scores + attention_mask
            )

        attn_weights = F.softmax(
            attn_scores.float(),
            dim=-1
        ).to(attn_scores.dtype)

        attn_weights = self.attn_dropout(
            attn_weights
        )

        # ----------------------------------------------------
        # Value is directly c_KV
        #
        # v_hat = c_KV
        # ----------------------------------------------------

        latent_output = torch.matmul(
            attn_weights,
            c_KV.unsqueeze(1).expand(
                -1,
                self.num_query_heads,
                -1,
                -1
            )
        )

        # ----------------------------------------------------
        # Absorb W_UV into output
        #
        # o_i = W_UV_i o'_i
        # ----------------------------------------------------

        W_UV = self.W_UV.weight

        W_UV = W_UV.view(
            self.num_kv_groups,
            self.head_dim,
            self.kv_latent_dim
        )

        W_UV_heads = W_UV[
            group_indices
        ]

        attn_output = torch.einsum(
            "bhtr,hdr->bhtd",
            latent_output,
            W_UV_heads
        )

        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        attn_output = (
            attn_output
            .transpose(1, 2)
            .contiguous()
            .view(
                B,
                T,
                self.num_query_heads * self.head_dim
            )
        )

        output = self.W_O(
            attn_output
        )

        # ----------------------------------------------------
        # Compact cache
        # ----------------------------------------------------

        if use_cache:

            new_cache = (
                c_KV.detach(),
                k_R.squeeze(1).detach()
            )

        else:

            new_cache = None

        return output, new_cache

    # ========================================================
    # Main forward
    # ========================================================

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        past_key_value=None,
        use_cache=False,
    ):
        """
        Parameters
        ----------
        hidden_states:
            [B, T, d_model]

        attention_mask:
            Optional additive attention mask.

        past_key_value:
            GQA path:
                (
                    k_C,
                    v_C,
                    k_R
                )

            MQA-absorb path:
                (
                    c_KV,
                    k_R
                )

            IMPORTANT:
                Set to None for every new independent sequence.

        use_cache:
            False during normal training.
            True during autoregressive generation.

        Returns
        -------
        output:
            [B, T, d_model]

        past_key_value:
            None when use_cache=False.
            Otherwise updated cache.
        """

        B, T, _ = hidden_states.shape

        # ----------------------------------------------------
        # Q latent
        # ----------------------------------------------------

        c_Q = self.W_DQ(hidden_states)

        # ----------------------------------------------------
        # KV latent
        # ----------------------------------------------------

        c_KV = self.W_DKV(hidden_states)

        # ----------------------------------------------------
        # q_R
        #
        # [B,T,dR]
        # ->
        # [B,1,T,dR]
        # ----------------------------------------------------

        q_R = self.W_QR(
            c_Q
        ).view(B, T, 1,self.rope_dim).transpose(1, 2)

        # ----------------------------------------------------
        # k_R
        # ----------------------------------------------------

        k_R_new = self.W_KR(
            hidden_states
        ).view(B, T, 1, self.rope_dim).transpose(1, 2)

        # ----------------------------------------------------
        # Select decoding path
        # ----------------------------------------------------

        if self.path == "gqa":

            return self._forward_gqa(
                hidden_states=hidden_states,
                c_Q=c_Q,
                c_KV=c_KV,
                q_R=q_R,
                k_R_new=k_R_new,
                past_key_value=past_key_value,
                attention_mask=attention_mask,
                use_cache=use_cache,
            )

        else:

            return self._forward_mqa_absorb(
                hidden_states=hidden_states,
                c_Q=c_Q,
                c_KV=c_KV,
                q_R=q_R,
                k_R_new=k_R_new,
                past_key_value=past_key_value,
                attention_mask=attention_mask,
                use_cache=use_cache,
            )

## GQLA Model Architecture

Here, we define the full GQLA language model. It comprises:

1.  **Token Embeddings**: Converts input token IDs into dense vector representations.
2.  **GQLABlock**: A stackable building block for the transformer. Each block includes:
    *   `GroupQueryLatentAttention`: The core attention mechanism with GQLA logic.
    *   **Feed-Forward Network**: A two-layer MLP with a GELU activation for processing the attention output.
    *   **Layer Normalization** and **Residual Connections**: Applied before and after the attention and feed-forward layers, respectively, to stabilize training.
    *   **Dropout**: For regularization.
3.  **Language Modeling Head**: A final linear layer that projects the output of the last GQLABlock to the vocabulary size, allowing for next-token prediction.

In [ ]:
class GQLABlock(nn.Module):

    def __init__(
        self,
        d_model: int,
        num_query_heads: int,
        num_kv_groups: int,
        head_dim: int,
        kv_latent_dim: int,
        rope_dim: int,
        q_latent_dim: int = None,
        dropout: float = 0.1,
        rope_base: float = 10000.0,
        path: str = "gqa",
        expansion_factor: int = 4,
    ):
        super().__init__()

        self.attn = GroupQueryLatentAttention(
            d_model=d_model,
            num_query_heads=num_query_heads,
            num_kv_groups=num_kv_groups,
            head_dim=head_dim,
            kv_latent_dim=kv_latent_dim,
            rope_dim=rope_dim,
            q_latent_dim=q_latent_dim,
            dropout=dropout,
            rope_base=rope_base,
            path=path,
        )

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, expansion_factor * d_model),
            nn.GELU(),
            nn.Linear(expansion_factor * d_model, d_model),
            nn.Dropout(dropout),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        past_key_value=None,
        use_cache=False,
    ):

        # Self-attention
        attn_output, new_past_key_value = self.attn(
            hidden_states=self.norm1(hidden_states),
            attention_mask=attention_mask,
            past_key_value=past_key_value,
            use_cache=use_cache,
        )

        hidden_states = hidden_states + self.dropout1(attn_output)

        # Feed-forward
        ff_output = self.feed_forward(self.norm2(hidden_states))
        hidden_states = hidden_states + self.dropout2(ff_output)

        return hidden_states, new_past_key_value


class GQLAModel(nn.Module):

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_layers: int,
        num_query_heads: int,
        num_kv_groups: int,
        head_dim: int,
        kv_latent_dim: int,
        rope_dim: int,
        q_latent_dim: int = None,
        dropout: float = 0.1,
        rope_base: float = 10000.0,
        path: str = "gqa",
        pad_token_id: int = 0,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=pad_token_id,
        )

        self.layers = nn.ModuleList([
            GQLABlock(
                d_model=d_model,
                num_query_heads=num_query_heads,
                num_kv_groups=num_kv_groups,
                head_dim=head_dim,
                kv_latent_dim=kv_latent_dim,
                rope_dim=rope_dim,
                q_latent_dim=q_latent_dim,
                dropout=dropout,
                rope_base=rope_base,
                path=path,
            )
            for _ in range(n_layers)
        ])

        self.norm_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(
        self,
        input_ids,
        attention_mask=None,
        past_key_values=None,
        use_cache=False,
    ):

        B, T = input_ids.shape

        hidden_states = self.token_embedding(input_ids)

        new_past_key_values = []

        for i, layer in enumerate(self.layers):

            past_key_value = (
                past_key_values[i]
                if past_key_values
                else None
            )

            hidden_states, new_past_key_value = layer(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
                past_key_value=past_key_value,
                use_cache=use_cache,
            )

            if use_cache:
                new_past_key_values.append(new_past_key_value)

        hidden_states = self.norm_final(hidden_states)
        logits = self.lm_head(hidden_states)

        if use_cache:
            return logits, new_past_key_values
        else:
            return logits

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int = None,
        eos_token_id: int = None,
    ):
        self.eval()
        generated_ids = input_ids.clone()
        past_key_values = None

        for _ in range(max_new_tokens):
            current_input_ids = generated_ids[:, -1:].to(input_ids.device) if generated_ids.shape[1] > 0 else input_ids[:, :1].to(input_ids.device)

            logits, past_key_values = self.forward(
                current_input_ids,
                past_key_values=past_key_values,
                use_cache=True,
            )

            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                top_k_logits, top_k_indices = torch.topk(logits, top_k)
                probs = F.softmax(top_k_logits, dim=-1)
                sampled = torch.multinomial(probs, num_samples=1)
                next_token_id = top_k_indices.gather(-1, sampled)
            else:
                next_token_id = torch.argmax(logits, dim=-1, keepdim=True)

            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

            if eos_token_id is not None and (next_token_id == eos_token_id).all():
                break

        return generated_ids


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GQLADataset(Dataset):

    def __init__(
        self,
        tokenized_documents,
        seq_len=512,
    ):
        self.documents = tokenized_documents
        self.seq_len = seq_len

        self.index = []

        for doc_idx, ids in enumerate(
            self.documents
        ):

            n = len(ids)

            if n < seq_len + 1:
                continue

            for start in range(
                0,
                n - seq_len,
                seq_len
            ):

                end = start + seq_len + 1

                if end <= n:
                    self.index.append(
                        (doc_idx, start, end)
                    )

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):

        doc_idx, start, end = self.index[idx]

        chunk = self.documents[
            doc_idx
        ][start:end]

        chunk = torch.tensor(
            chunk,
            dtype=torch.long
        )

        return {
            "input_ids": chunk[:-1],
            "labels": chunk[1:],
        }

In [ ]:
from torch.utils.data import DataLoader


seq_len = 512
batch_size = 4 # Reduced from 8

train_dataset = GQLADataset(
    train_tokens,
    seq_len=seq_len,
)

val_dataset = GQLADataset(
    val_tokens,
    seq_len=seq_len,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)


del train_dataset, val_dataset

NameError: name 'train_tokens' is not defined

In [ ]:
import torch

def validate_gqla(
    model,
    dataloader,
    device="cuda",
    use_amp=True,
):
    """
    Evaluates the model on the validation dataloader.
    """
    model.eval()
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(use_amp and device == "cuda")
            ):
                outputs = model(
                    input_ids,
                    use_cache=False,
                )

                if isinstance(outputs, tuple):
                    logits = outputs[0]
                else:
                    logits = outputs

                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = labels[:, 1:].contiguous()

                loss = F.cross_entropy(
                    shift_logits.view(
                        -1,
                        shift_logits.size(-1)
                    ),
                    shift_labels.view(-1),
                )

            total_loss += loss.item()
            num_batches += 1

    avg_loss = total_loss / num_batches if num_batches > 0 else float('inf')
    model.train()
    return avg_loss

In [ ]:
# import time

# def train_gqla(
#     model,
#     dataloader,
#     val_dataloader, # Added validation dataloader
#     optimizer,
#     scheduler=None,
#     device="cuda",
#     epochs=1,
#     max_steps=None,
#     grad_accum_steps=1,
#     grad_clip=1.0,
#     checkpoint_dir=None,
#     save_every=1000,
#     log_every=100,
#     tag="checkpoint", # New parameter
#     use_amp=True,
# ):
#     """
#     Generic GQLA language-model training loop.

#     Expected dataloader output:
#         input_ids: [B, T]
#         labels:    [B, T]

#     Expected model output:
#         logits: [B, T, vocab_size]

#     If your TransformerXL model also returns memory, this loop
#     can easily be extended to carry it.
#     """

#     os.makedirs(checkpoint_dir, exist_ok=True) if checkpoint_dir else None

#     # Initialize training state variables
#     start_epoch = 0
#     global_step = 0
#     best_val_loss = float('inf') # Track best validation loss
#     loss_history = []

#     load_from_checkpoint_path = os.path.join(checkpoint_dir, f"{tag}.pt") if checkpoint_dir else None
#     print(f"Loading from checkpoint: {load_from_checkpoint_path}")

#     # Load state from checkpoint if path is provided
#     if load_from_checkpoint_path and os.path.exists(load_from_checkpoint_path):
#         (
#             loaded_epoch,
#             loaded_last_loss,
#             loaded_best_val_loss, # Load best_val_loss
#             loaded_loss_history,
#             loaded_global_step
#         ) = load_checkpoint(load_from_checkpoint_path, model, optimizer)

#         start_epoch = loaded_epoch
#         global_step = loaded_global_step
#         best_val_loss = loaded_best_val_loss # Assign to best_val_loss
#         loss_history = loaded_loss_history
#         print(f"Resuming training from Epoch {start_epoch}, Global Step {global_step} with best recorded validation loss {best_val_loss:.4f}")
#     elif load_from_checkpoint_path:
#         print(f"Warning: Checkpoint path '{load_from_checkpoint_path}' not found. Starting training from scratch.")


#     model.train()

#     scaler = torch.amp.GradScaler(
#         "cuda",
#         enabled=(use_amp and device == "cuda")
#     )

#     running_loss = 0.0
#     tokens_seen = 0

#     start_time = time.time()

#     for epoch in range(start_epoch, epochs):

#         epoch_loss = 0.0
#         epoch_tokens = 0
#         epoch_start = time.time()

#         optimizer.zero_grad(set_to_none=True)

#         for batch_idx, batch in enumerate(dataloader):

#             input_ids = batch["input_ids"].to(
#                 device,
#                 non_blocking=True
#             )
#             labels = batch["labels"].to(
#                 device,
#                 non_blocking=True
#             )

#             B, T = input_ids.shape

#             # -----------------------------
#             # Forward
#             # -----------------------------

#             with torch.autocast(
#                 device_type=device.type,
#                 dtype=torch.float16,
#                 enabled=(use_amp and device.type == "cuda")
#             ):

#                 outputs = model(
#                     input_ids,
#                     use_cache=False,
#                 )

#                 logits = outputs[0] if isinstance(outputs, tuple) else outputs

#                 shift_logits = logits[:, :-1, :].contiguous()
#                 shift_labels = labels[:, 1:].contiguous()

#                 loss = F.cross_entropy(
#                     shift_logits.view(-1, shift_logits.size(-1)),
#                     shift_labels.view(-1),
#                 )

#                 # Normalize for gradient accumulation
#                 loss_for_backward = loss / grad_accum_steps

#             # -----------------------------
#             # Backward
#             # -----------------------------

#             scaler.scale(loss_for_backward).backward()

#             running_loss += loss.item()
#             epoch_loss += loss.item()

#             batch_tokens = B * T
#             tokens_seen += batch_tokens
#             epoch_tokens += batch_tokens

#             # -----------------------------
#             # Optimizer step
#             # -----------------------------

#             should_step = (
#                 (batch_idx + 1) % grad_accum_steps == 0
#                 or (batch_idx + 1) == len(dataloader)
#             )

#             if should_step:

#                 scaler.unscale_(optimizer)

#                 grad_norm = torch.nn.utils.clip_grad_norm_(
#                     model.parameters(),
#                     grad_clip
#                 )

#                 scaler.step(optimizer)
#                 scaler.update()

#                 optimizer.zero_grad(set_to_none=True)

#                 if scheduler is not None:
#                     scheduler.step()

#                 global_step += 1

#                 # -----------------------------
#                 # Logging
#                 # -----------------------------

#                 if global_step % log_every == 0:

#                     avg_loss = running_loss / log_every

#                     ppl = math.exp(min(avg_loss, 20))

#                     elapsed = time.time() - start_time

#                     tok_per_sec = (
#                         tokens_seen / max(elapsed, 1e-6)
#                     )

#                     print(
#                         f"step={global_step:,} | "
#                         f"batch={batch_idx + 1:,} | "
#                         f"loss={avg_loss:.4f} | "
#                         f"ppl={ppl:.2f} | "
#                         f"tokens={tokens_seen / 1e6:.2f}M | "
#                         f"tok/s={tok_per_sec:,.0f} | "
#                         f"grad_norm={grad_norm:.3f}",
#                         flush=True
#                     )

#                     running_loss = 0.0
#                     tokens_seen = 0

#                 # -----------------------------
#                 # Max steps
#                 # -----------------------------

#                 if (
#                     max_steps is not None
#                     and global_step >= max_steps
#                 ):
#                     print(
#                         f"Reached max_steps={max_steps:,}",
#                         flush=True
#                     )
#                     return global_step

#         # --------------------------------------------------------
#         # Epoch summary and validation
#         # --------------------------------------------------------

#         avg_epoch_train_loss = (
#             epoch_loss /
#             max(batch_idx + 1, 1)
#         )

#         # Run validation
#         avg_epoch_val_loss = validate_gqla(
#             model=model,
#             dataloader=val_dataloader,
#             device=device,
#             use_amp=use_amp,
#         )

#         loss_history.append({'train_loss': avg_epoch_train_loss, 'val_loss': avg_epoch_val_loss}) # Update loss_history with both train and val loss

#         # Update global best_val_loss and save the best model
#         if avg_epoch_val_loss < best_val_loss:
#             best_val_loss = avg_epoch_val_loss
#             print(f"New best epoch validation loss: {best_val_loss:.4f}!")
#             # Save the best model
#             if checkpoint_dir is not None:
#                 best_ckpt_path = os.path.join(
#                     checkpoint_dir,
#                     "best_model.pt"
#                 )
#                 save_checkpoint(
#                     model,
#                     optimizer,
#                     epoch=epoch + 1, # Saving the epoch when best model was found
#                     loss=avg_epoch_train_loss, # Saving train loss of that epoch
#                     filepath=best_ckpt_path,
#                     best_loss=best_val_loss, # Saving best validation loss
#                     loss_history=loss_history,
#                     global_step=global_step # Ensure global_step is saved here too
#                 )
#                 print(f"Best model saved to: {best_ckpt_path}")

#         # Always save an end-of-epoch checkpoint for resuming training
#         if checkpoint_dir is not None:
#             save_to_checkpoint_path = load_from_checkpoint_path

#             save_checkpoint(
#                 model,
#                 optimizer,
#                 epoch=epoch + 1,
#                 loss=avg_epoch_train_loss, # Saving train loss for current epoch
#                 filepath=load_from_checkpoint_path,
#                 best_loss=best_val_loss, # Saving best validation loss
#                 loss_history=loss_history,
#                 global_step=global_step # Ensure global_step is saved here too
#             )
#             print(f"End-of-epoch checkpoint saved: {save_to_checkpoint_path}")

#         epoch_ppl = math.exp(
#             min(avg_epoch_train_loss, 20)
#         )

#         elapsed = time.time() - epoch_start

#         print(
#             "\n"
#             f"Epoch {epoch + 1} complete\n"
#             f"Train Loss      : {avg_epoch_train_loss:.4f}\n"
#             f"Validation Loss : {avg_epoch_val_loss:.4f}\n"
#             f"Perplexity: {epoch_ppl:.2f}\n"
#             f"Tokens    : {epoch_tokens/1e6:.2f}M\n"
#             f"Time      : {elapsed/60:.2f} min\n"
#             f"Steps     : {global_step:,}\n"
#         )

#     return global_step

In [ ]:
import os
import time
import math
import torch
import torch.nn.functional as F


def train_gqla(
    model,
    dataloader,
    val_dataloader,
    optimizer,
    scheduler=None,
    device="cuda",
    epochs=1,
    max_steps=None,
    grad_accum_steps=1,
    grad_clip=1.0,
    checkpoint_dir=None,
    save_every=1000,
    log_every=100,
    tag="checkpoint",
    use_amp=True,
):
    """
    Train GQLA with periodic and end-of-epoch checkpoints.

    Periodic checkpoint filenames:
        {tag}_epoch{epoch}_step{global_step}.pt

    Latest resumable checkpoint:
        {tag}.pt

    Best validation checkpoint:
        best_model.pt

    Checkpoints include:
        model, optimizer, scheduler, AMP scaler,
        epoch, batch position, global step,
        best validation loss, and loss history.
    """

    device = torch.device(device)
    model.to(device)

    if checkpoint_dir is not None:
        os.makedirs(checkpoint_dir, exist_ok=True)

    if grad_accum_steps < 1:
        raise ValueError("grad_accum_steps must be >= 1")

    if log_every < 1:
        raise ValueError("log_every must be >= 1")

    # --------------------------------------------------
    # Paths
    # --------------------------------------------------

    latest_path = (
        os.path.join(checkpoint_dir, f"{tag}.pt")
        if checkpoint_dir is not None
        else None
    )

    # --------------------------------------------------
    # Training state
    # --------------------------------------------------

    start_epoch = 0
    start_batch = 0
    global_step = 0
    best_val_loss = float("inf")
    loss_history = []

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(use_amp and device.type == "cuda"),
    )

    # --------------------------------------------------
    # Resume checkpoint
    # --------------------------------------------------

    if latest_path is not None and os.path.isfile(latest_path):
        print(f"Loading checkpoint: {latest_path}")

        ckpt = torch.load(
            latest_path,
            map_location=device,
            weights_only=False,
        )

        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])

        if scheduler is not None and ckpt.get("scheduler_state_dict") is not None:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])

        if ckpt.get("scaler_state_dict") is not None:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

        start_epoch = ckpt.get("epoch", 0)
        start_batch = ckpt.get("batch_idx", 0)
        global_step = ckpt.get("global_step", 0)
        best_val_loss = ckpt.get("best_val_loss", float("inf"))
        loss_history = ckpt.get("loss_history", [])

        # Restore RNG states when available.
        if ckpt.get("torch_rng_state") is not None:
            torch.set_rng_state(ckpt["torch_rng_state"].cpu().to(dtype=torch.uint8))

        if (
            device.type == "cuda"
            and ckpt.get("cuda_rng_state") is not None
        ):
            # Ensure each state tensor is explicitly cast to a CPU ByteTensor (uint8)
            cleaned_cuda_rng_state = [
                state.cpu().to(dtype=torch.uint8) for state in ckpt["cuda_rng_state"]
            ]
            torch.cuda.set_rng_state_all(cleaned_cuda_rng_state)

        print(
            f"Resuming from epoch={start_epoch + 1}, "
            f"batch={start_batch}, step={global_step}, "
            f"best_val_loss={best_val_loss:.4f}"
        )

    # --------------------------------------------------
    # Checkpoint helper
    # --------------------------------------------------

    def save_training_checkpoint(
        path,
        epoch_to_resume,
        batch_to_resume,
        last_loss,
    ):
        checkpoint = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": (
                scheduler.state_dict()
                if scheduler is not None
                else None
            ),
            "scaler_state_dict": scaler.state_dict(),

            # Zero-based epoch and next batch to process.
            "epoch": epoch_to_resume,
            "batch_idx": batch_to_resume,
            "global_step": global_step,

            "loss": last_loss,
            "best_val_loss": best_val_loss,
            "loss_history": loss_history,

            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if device.type == "cuda"
                else None
            ),
        }

        # Write atomically to avoid leaving a partial checkpoint.
        temp_path = path + ".tmp"
        torch.save(checkpoint, temp_path)
        os.replace(temp_path, path)

    # --------------------------------------------------
    # Training
    # --------------------------------------------------

    model.train()

    running_loss = 0.0
    running_batches = 0
    tokens_seen = 0
    start_time = time.time()

    last_loss = float("nan")

    for epoch in range(start_epoch, epochs):

        # Only skip batches when resuming the interrupted epoch.
        batch_offset = start_batch if epoch == start_epoch else 0

        epoch_start = time.time()
        epoch_loss = 0.0
        epoch_batches = 0
        epoch_tokens = 0

        optimizer.zero_grad(set_to_none=True)

        for batch_idx, batch in enumerate(dataloader):

            # Resume at the next unprocessed batch.
            if batch_idx < batch_offset:
                continue

            input_ids = batch["input_ids"].to(
                device, non_blocking=True
            )
            labels = batch["labels"].to(
                device, non_blocking=True
            )

            _, seq_len = input_ids.shape
            batch_size = input_ids.size(0)

            # ------------------------------------------
            # Forward and loss
            # ------------------------------------------

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(use_amp and device.type == "cuda"),
            ):
                outputs = model(input_ids, use_cache=False)

                logits = (
                    outputs[0]
                    if isinstance(outputs, tuple)
                    else outputs
                )

                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = labels[:, 1:].contiguous()

                loss = F.cross_entropy(
                    shift_logits.reshape(
                        -1, shift_logits.size(-1)
                    ),
                    shift_labels.reshape(-1),
                    ignore_index=-100,
                )

            last_loss = loss.item()

            # ------------------------------------------
            # Backward
            # ------------------------------------------

            (loss / grad_accum_steps).backward() \
                if not scaler.is_enabled() else None

            if scaler.is_enabled():
                scaler.scale(loss / grad_accum_steps).backward()

            running_loss += last_loss
            running_batches += 1
            epoch_loss += last_loss
            epoch_batches += 1

            batch_tokens = batch_size * seq_len
            tokens_seen += batch_tokens
            epoch_tokens += batch_tokens

            # ------------------------------------------
            # Optimizer step
            # ------------------------------------------

            is_last_batch = (batch_idx + 1) == len(dataloader)

            should_step = (
                (batch_idx + 1) % grad_accum_steps == 0
                or is_last_batch
            )

            if not should_step:
                continue

            if scaler.is_enabled():
                scaler.unscale_(optimizer)

            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                grad_clip,
            )

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None:
                scheduler.step()

            global_step += 1

            # ------------------------------------------
            # Periodic step checkpoint
            # ------------------------------------------

            if (
                checkpoint_dir is not None
                and save_every is not None
                and save_every > 0
                and global_step % save_every == 0
            ):
                step_path = os.path.join(
                    checkpoint_dir,
                    f"{tag}_epoch{epoch + 1}_step{global_step}.pt",
                )

                # Save both the archival checkpoint and
                # the latest resumable checkpoint.

                save_training_checkpoint(
                    latest_path,
                    epoch,
                    batch_idx + 1,
                    last_loss,
                )

                print(f"Saved step checkpoint: {step_path}")

            # ------------------------------------------
            # Logging
            # ------------------------------------------

            if global_step % log_every == 0:
                avg_loss = running_loss / max(running_batches, 1)
                ppl = math.exp(min(avg_loss, 20))

                elapsed = time.time() - start_time
                tok_per_sec = tokens_seen / max(elapsed, 1e-6)

                print(
                    f"epoch={epoch + 1} | "
                    f"step={global_step:,} | "
                    f"batch={batch_idx + 1:,} | "
                    f"loss={avg_loss:.4f} | "
                    f"ppl={ppl:.2f} | "
                    f"tokens={tokens_seen / 1e6:.2f}M | "
                    f"tok/s={tok_per_sec:,.0f} | "
                    f"grad_norm={grad_norm.item():.3f}",
                    flush=True,
                )

                running_loss = 0.0
                running_batches = 0
                tokens_seen = 0
                start_time = time.time()

            # ------------------------------------------
            # Max steps
            # ------------------------------------------

            if max_steps is not None and global_step >= max_steps:

                # Persist state before returning.
                save_training_checkpoint(
                    latest_path,
                    epoch,
                    batch_idx + 1,
                    last_loss,
                ) if latest_path is not None else None

                print(f"Reached max_steps={max_steps}")
                return global_step

        # ----------------------------------------------
        # Epoch summary
        # ----------------------------------------------

        avg_train_loss = epoch_loss / max(epoch_batches, 1)

        avg_val_loss = validate_gqla(
            model=model,
            dataloader=val_dataloader,
            device=device,
            use_amp=use_amp,
        )

        loss_history.append({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
        })

        # ----------------------------------------------
        # Best model
        # ----------------------------------------------

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss

            if checkpoint_dir is not None:
                best_path = os.path.join(
                    checkpoint_dir, "best_model.pt"
                )

                save_training_checkpoint(
                    best_path,
                    epoch + 1,
                    0,
                    avg_train_loss,
                )

                print(
                    f"New best validation loss: "
                    f"{best_val_loss:.4f}"
                )

        # ----------------------------------------------
        # End-of-epoch checkpoint
        # ----------------------------------------------

        if latest_path is not None:
            save_training_checkpoint(
                latest_path,
                epoch + 1,
                0,
                avg_train_loss,
            )

            print(f"Saved epoch checkpoint: {latest_path}")

        epoch_ppl = math.exp(min(avg_train_loss, 20))

        print(
            f"\nEpoch {epoch + 1} complete\n"
            f"Train Loss      : {avg_train_loss:.4f}\n"
            f"Validation Loss : {avg_val_loss:.4f}\n"
            f"Perplexity      : {epoch_ppl:.2f}\n"
            f"Tokens          : {epoch_tokens / 1e6:.2f}M\n"
            f"Time            : "
            f"{(time.time() - epoch_start) / 60:.2f} min\n"
            f"Global Step     : {global_step:,}\n",
            flush=True,
        )

        start_batch = 0

    return global_step


## Model Initialization and Training

Now, we initialize the `GQLAModel` with specified hyperparameters and an `AdamW` optimizer. Then, we initiate the training process using the `train_gqla` function.

**Model Parameters:**
-   `vocab_size`: The size of the vocabulary from our BPE tokenizer.
-   `d_model`: The dimension of the model's embeddings and internal representations.
-   `n_layers`: The number of GQLA blocks (transformer layers).
-   `num_query_heads`: The number of attention heads for queries.
-   `num_kv_groups`: The number of groups for KV attention (used in GQLA).
-   `head_dim`: The dimension of each attention head.
-   `kv_latent_dim`: The latent dimension for KV projections.
-   `rope_dim`: The dimension for Rotary Positional Embeddings.
-   `path`: The GQLA attention path ('gqa' or 'mqa_absorb').

**Training Parameters:**
-   `epochs`: Number of training epochs.
-   `grad_accum_steps`: Number of gradient accumulation steps.
-   `learning_rate`: The initial learning rate for the optimizer.
-   `weight_decay`: L2 regularization for the optimizer.
-   `checkpoint_dir`: Directory to save model checkpoints.
-   `save_every`: Frequency (in steps) to save checkpoints.
-   `log_every`: Frequency (in steps) to log training progress.

In [ ]:
import gc
import torch

# Delete old objects if they exist
for name in [
    "model",
    "optimizer",
    "scheduler",
    "scaler",
    "loss",
    "outputs",
    "logits",
    "batch",
    "input_ids",
    "labels",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(
    f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB"
)
print(
    f"Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB"
)



Allocated: 0.00 GB
Reserved: 0.00 GB


In [ ]:
# from torch.optim import AdamW
# import datetime

# # Model Parameters
# vocab_size = len(tokenizer.itos)
# d_model = 384 # Reduced from 256
# n_layers = 6 # Reduced from 32
# num_query_heads = 12 # Reduced from 64
# num_kv_groups = 4 # Reduced from 32 (ensure num_query_heads % num_kv_groups == 0)
# head_dim = 32 # Reduced from 32
# kv_latent_dim = 96 # Reduced from 128
# rope_dim = 16 # Reduced from 16
# path = "gqa" # Or "mqa_absorb"
# pad_token_id = tokenizer.stoi["<pad>"]

# # Training Parameters
# epochs = 50
# grad_accum_steps = 4
# learning_rate = 1e-4
# weight_decay = 0.01
# grad_clip = 1.0
# save_every = 250
# log_every = 50

# # Checkpoint directory
# # Use a timestamp to create a unique checkpoint directory for each run
# timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
# checkpoint_dir = os.path.join(cache_dir, f"gqla_checkpoints")
# os.makedirs(checkpoint_dir, exist_ok=True)
# print(f"Checkpoints will be saved to: {checkpoint_dir}")



In [ ]:
from torch.optim import AdamW
import datetime

# Model Parameters
vocab_size = len(tokenizer.itos)
d_model = 512 # Reduced from 256
n_layers = 10 # Reduced from 32
num_query_heads = 64 # Reduced from 64
num_kv_groups = 32 # Reduced from 32 (ensure num_query_heads % num_kv_groups == 0)
head_dim = 64 # Reduced from 32
kv_latent_dim = 96 # Reduced from 128
rope_dim = 32 # Reduced from 16
path = "gqa" # Or "mqa_absorb"
pad_token_id = tokenizer.stoi["<pad>"]

# Training Parameters
epochs = 3
grad_accum_steps = 16
learning_rate = 1e-4
weight_decay = 0.01
grad_clip = 1.0
save_every = 250
log_every = 50

# Checkpoint directory
# Use a timestamp to create a unique checkpoint directory for each run
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
checkpoint_dir = os.path.join(cache_dir, f"gqla_checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Checkpoints will be saved to: {checkpoint_dir}")



Checkpoints will be saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints


In [ ]:


# Instantiate the model
model = GQLAModel(
    vocab_size=vocab_size,
    d_model=d_model,
    n_layers=n_layers,
    num_query_heads=num_query_heads,
    num_kv_groups=num_kv_groups,
    head_dim=head_dim,
    kv_latent_dim=kv_latent_dim,
    rope_dim=rope_dim,
    path=path,
    pad_token_id=pad_token_id,
).to(device)

print(f"Model initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")

# Instantiate the optimizer
optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

Model initialized with 103,888,896 trainable parameters.


In [ ]:
# Start training
print("Starting GQLA model training...")


final_step = train_gqla(
    model=model,
    dataloader=train_loader,
    val_dataloader=val_loader, # Added val_dataloader
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    grad_accum_steps=grad_accum_steps,
    checkpoint_dir=checkpoint_dir,
    save_every=save_every,
    log_every=log_every,
    use_amp=False,
    tag="gqla_baseline_large"
)

print(f"Training finished at global step: {final_step:,}")

Starting GQLA model training...
epoch=1 | step=50 | batch=800 | loss=8.0006 | ppl=2982.79 | tokens=1.64M | tok/s=2,391 | grad_norm=0.755
epoch=1 | step=100 | batch=1,600 | loss=6.7328 | ppl=839.47 | tokens=1.64M | tok/s=2,447 | grad_norm=0.553
epoch=1 | step=150 | batch=2,400 | loss=6.4627 | ppl=640.81 | tokens=1.64M | tok/s=2,451 | grad_norm=0.358
epoch=1 | step=200 | batch=3,200 | loss=6.3059 | ppl=547.82 | tokens=1.64M | tok/s=2,443 | grad_norm=0.402
Saved step checkpoint: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/gqla_baseline_large_epoch1_step250.pt
epoch=1 | step=250 | batch=4,000 | loss=6.1452 | ppl=466.45 | tokens=1.64M | tok/s=2,423 | grad_norm=0.561
epoch=1 | step=300 | batch=4,800 | loss=6.0266 | ppl=414.28 | tokens=1.64M | tok/s=2,459 | grad_norm=0.463
epoch=1 | step=350 | batch=5,600 | loss=5.9553 | ppl=385.78 | tokens=1.64M | tok/s=2,455 | grad_norm=0.606
epoch=1 | step=400 | batch=6,400 | loss=5.8822 | ppl=358.61 | tokens=1.64M | 

In [ ]:
# Start training
print("Starting GQLA model training...")


final_step = train_gqla(
    model=model,
    dataloader=train_loader,
    val_dataloader=val_loader, # Added val_dataloader
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    grad_accum_steps=grad_accum_steps,
    checkpoint_dir=checkpoint_dir,
    save_every=save_every,
    log_every=log_every,
    use_amp=False,
    tag="gqla_baseline_large"
)

print(f"Training finished at global step: {final_step:,}")

Starting GQLA model training...
Loading checkpoint: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/gqla_baseline_large.pt
Resuming from epoch=1, batch=68000, step=4250, best_val_loss=inf
epoch=1 | step=4,300 | batch=68,800 | loss=4.4335 | ppl=84.23 | tokens=1.64M | tok/s=1,864 | grad_norm=0.536
epoch=1 | step=4,350 | batch=69,600 | loss=4.4177 | ppl=82.90 | tokens=1.64M | tok/s=2,077 | grad_norm=0.532
epoch=1 | step=4,400 | batch=70,400 | loss=4.4058 | ppl=81.92 | tokens=1.64M | tok/s=2,069 | grad_norm=0.536
epoch=1 | step=4,450 | batch=71,200 | loss=4.4074 | ppl=82.05 | tokens=1.64M | tok/s=2,068 | grad_norm=0.547
Saved step checkpoint: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/gqla_baseline_large_epoch1_step4500.pt
epoch=1 | step=4,500 | batch=72,000 | loss=4.4303 | ppl=83.95 | tokens=1.64M | tok/s=2,062 | grad_norm=0.551
epoch=1 | step=4,550 | batch=72,800 | loss=4.4016 | ppl=81.58 | tokens=1.64M | tok/s=2,

In [ ]:
# Start training
print("Starting GQLA model training...")


final_step = train_gqla(
    model=model,
    dataloader=train_loader,
    val_dataloader=val_loader, # Added val_dataloader
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    grad_accum_steps=grad_accum_steps,
    checkpoint_dir=checkpoint_dir,
    save_every=save_every,
    log_every=log_every,
    use_amp=False,
    tag="gqla_baseline_large"
)

print(f"Training finished at global step: {final_step:,}")

Starting GQLA model training...
Loading checkpoint: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/gqla_baseline_large.pt
Resuming from epoch=1, batch=168000, step=10500, best_val_loss=inf
epoch=1 | step=10,550 | batch=168,800 | loss=4.0217 | ppl=55.80 | tokens=1.64M | tok/s=1,711 | grad_norm=0.570
epoch=1 | step=10,600 | batch=169,600 | loss=4.0044 | ppl=54.84 | tokens=1.64M | tok/s=2,209 | grad_norm=0.553
epoch=1 | step=10,650 | batch=170,400 | loss=3.9744 | ppl=53.22 | tokens=1.64M | tok/s=2,212 | grad_norm=0.566
epoch=1 | step=10,700 | batch=171,200 | loss=4.0030 | ppl=54.76 | tokens=1.64M | tok/s=2,215 | grad_norm=0.557
Saved step checkpoint: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/gqla_baseline_large_epoch1_step10750.pt
epoch=1 | step=10,750 | batch=172,000 | loss=4.0253 | ppl=56.00 | tokens=1.64M | tok/s=2,196 | grad_norm=0.577
epoch=1 | step=10,800 | batch=172,800 | loss=3.9919 | ppl=54.16 | tokens=1

In [ ]:
test_file_path = hf_hub_download(
    repo_id="Exploration-Lab/CS-781-Capstone",
    filename="Phase_1_Pretraining/Test/test.json",
    repo_type="dataset",
    token=hf_token
)

print("Test file:", test_file_path)

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Test file: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_1_Pretraining/Test/test.json


In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


In [ ]:
#!/usr/bin/env python3
"""Local evaluator for Phase 1.

Usage:
  python evaluate_phase1.py --truth truth.json --prediction prediction.json

Requires: bert-score and torch for the official primary metric.
"""
import argparse, json, math, sys
from collections import Counter

try:
    import bert_score
except ImportError as exc:
    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc

class DuplicateJSONKeyError(ValueError): pass

def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")

def pairs_no_duplicates(pairs):
    out = {}; dup = []
    for k, v in pairs:
        if k in out: dup.append(k)
        out[k] = v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out

def load_json(path):
    try:
        with open(path, encoding='utf-8') as f:
            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e

def validate(data, name):
    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")
    for i, rec in data.items():
        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or 'text' not in rec or not isinstance(rec['text'], str) or not rec['text'].strip():
            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field 'text'.")

def lcs_length(a,b):
    if not a or not b: return 0
    if len(a) < len(b): a,b=b,a
    masks={}
    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)
    state=0
    for t in b:
        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)
    return state.bit_count() if hasattr(int,'bit_count') else bin(state).count('1')

def rouge_l_f1(p,r):
    pt=p.split(); rt=r.split()
    if not pt or not rt: return 0.0
    l=lcs_length(pt,rt)
    if not l: return 0.0
    pp=l/len(pt); rr=l/len(rt)
    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)

def ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()

def bleu4(preds, refs):
    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0
    for p,r in zip(preds,refs):
        pl+=len(p); rl+=len(r)
        for n in range(1,order+1):
            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())
            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))
    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]
    if any(x<=0 for x in prec) or pl==0: return 0.0
    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)
    return bp*math.exp(sum(math.log(x) for x in prec)/order)

def run_evaluation(truth_path, prediction_path):
    truth = load_json(truth_path)
    pred = load_json(prediction_path)
    validate(truth, 'truth')
    validate(pred, 'prediction')

    if set(truth.keys()) != set(pred.keys()):
        # Find missing and extra keys for better error message
        missing_keys = set(truth.keys()) - set(pred.keys())
        extra_keys = set(pred.keys()) - set(truth.keys())
        error_msg = f"ID mismatch: "
        if missing_keys: error_msg += f"missing={len(missing_keys)} (e.g., {list(missing_keys)[:5]}), "
        if extra_keys: error_msg += f"extra={len(extra_keys)} (e.g., {list(extra_keys)[:5]}), "
        raise ValueError(error_msg.strip(', '))

    # Ensure consistent ordering for predictions and references based on truth keys
    ids_ordered = list(truth.keys())

    ps = [pred[i]['text'].split() for i in ids_ordered]
    rs = [truth[i]['text'].split() for i in ids_ordered]
    pred_texts = [pred[i]['text'].strip() for i in ids_ordered]
    ref_texts = [truth[i]['text'].strip() for i in ids_ordered]

    try:
        import torch
        device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError:
        device='cpu'
        print("PyTorch not found or CUDA not available, using CPU for BERTScore.")

    print("Calculating BERTScore...")
    _,_,f = bert_score.score(pred_texts, ref_texts, lang='en', batch_size=8, verbose=False, **({'device':device} if device else {}))
    bertscore = float(f.mean().item())
    print(f"BERTScore: {bertscore:.4f}")

    print("Calculating ROUGE-L...")
    rouge = sum(rouge_l_f1(pred[i]['text'],truth[i]['text']) for i in ids_ordered)/len(ids_ordered)
    print(f"ROUGE-L: {rouge:.4f}")

    print("Calculating BLEU-4...")
    bleu = bleu4(ps, rs)
    print(f"BLEU-4: {bleu:.4f}")

    return {
        "bertscore": bertscore,
        "rouge_l": rouge,
        "bleu_4": bleu
    }

# The main function is removed as this script will be imported and its functions called directly.


### Load Best Model and Prepare for Prediction

First, I will load the best performing model based on validation loss from the saved checkpoint `best_model.pt`.

In [ ]:
import torch
import os, gc

# Clear GPU memory before loading the model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

checkpoint_dir = os.path.join(cache_dir, f"gqla_checkpoints")
# Define the path to the best model checkpoint
best_model_path = os.path.join(checkpoint_dir, "gqla_baseling_large.pt")

model = GQLAModel(
    vocab_size=vocab_size,
    d_model=d_model,
    n_layers=n_layers,
    num_query_heads=num_query_heads,
    num_kv_groups=num_kv_groups,
    head_dim=head_dim,
    kv_latent_dim=kv_latent_dim,
    rope_dim=rope_dim,
    path=path,
    pad_token_id=pad_token_id,
).to(device)


# Load the best model
# We don't need the optimizer state for inference
_, _, _, _, _ = load_checkpoint(best_model_path, model)
model.eval()

print("Best model loaded successfully for prediction.")

No checkpoint found at /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/gqla_checkpoints/gqla_baseling_large.pt
Best model loaded successfully for prediction.


### Generate Predictions and Ground Truth

Now, I will iterate through the `test_data`, tokenize each example, split it into a prompt and a target continuation, generate text based on the prompt using the loaded model, and then decode both the generated text and the true target continuation.

These will be stored in `predictions_output` and `truth_output` dictionaries, mapping the example ID to the generated/true text.

In [ ]:
CONTEXT_LEN = 512
TARGET_LEN = 128
TOTAL_LEN = CONTEXT_LEN + TARGET_LEN

In [ ]:
# Initialize dictionaries to store predictions and ground truth
predictions_output = {}
truth_output = {}

max_seq_len = CONTEXT_LEN # Use the same sequence length as training
generation_prompt_len = TARGET_LEN // 2 # Generate for the second half of the sequence

print(f"Generating predictions for {len(test_data)} test examples...")

example_no = 1

for example_id, example_data in list(test_data.items()):

    if example_no % 100 == 0:
      print(f"Processing example {example_no}/{len(test_data)}: {example_id}")
    example_no += 1
    input_text = example_data['text']

    token_ids = tokenizer.encode(
        input_text,
        add_special_tokens=True
    )

    if len(token_ids) < TOTAL_LEN:
        token_ids.extend(
            [pad_token_id] * (TOTAL_LEN - len(token_ids))
        )
    else:
        token_ids = token_ids[:TOTAL_LEN]

    prompt_ids = token_ids[:CONTEXT_LEN]
    target_ids = token_ids[CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN]

    prompt_tensor = torch.tensor(
        prompt_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    generated_ids = model.generate(
        input_ids=prompt_tensor,
        max_new_tokens=TARGET_LEN,
        temperature=1.0,
        top_k=50,
        eos_token_id=tokenizer.stoi['</s>']
    )

    generated_continuation_ids = generated_ids[
        0, CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN
    ].tolist()

    predicted_text = tokenizer.decode(generated_continuation_ids)
    true_text = tokenizer.decode(target_ids)
    predictions_output[example_id] = {'text': predicted_text}
    truth_output[example_id] = {'text': true_text}

import json

prediction_json_path = os.path.join(cache_dir, "prediction_small.json")
truth_json_path = os.path.join(cache_dir, "truth_small.json")

with open(prediction_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions_output, f, ensure_ascii=False, indent=2)
print(f"Predictions saved to: {prediction_json_path}")

with open(truth_json_path, "w", encoding="utf-8") as f:
    json.dump(truth_output, f, ensure_ascii=False, indent=2)
print(f"Ground truth saved to: {truth_json_path}")

evaluation_results = run_evaluation(truth_json_path, prediction_json_path)

print("\nEvaluation Results:")
for metric, score in evaluation_results.items():
    print(f"- {metric.replace('_', '-').upper()}: {score:.4f}")



print(f"Generated predictions for {len(predictions_output)} examples.")

Generating predictions for 1000 test examples...
Processing example 100/1000: 100
Processing example 200/1000: 200
Processing example 300/1000: 300
Processing example 400/1000: 400
Processing example 500/1000: 500
Processing example 600/1000: 600
Processing example 700/1000: 700
Processing example 800/1000: 800
Processing example 900/1000: 900
Processing example 1000/1000: 1000
Predictions saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/prediction_small.json
Ground truth saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/truth_small.json
Calculating BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.7321
Calculating ROUGE-L...
ROUGE-L: 0.0018
Calculating BLEU-4...
BLEU-4: 0.0000

Evaluation Results:
- BERTSCORE: 0.7321
- ROUGE-L: 0.0018
- BLEU-4: 0.0000
Generated predictions for 1000 examples.


In [ ]:
# Initialize dictionaries to store predictions and ground truth
predictions_output = {}
truth_output = {}

max_seq_len = CONTEXT_LEN # Use the same sequence length as training
generation_prompt_len = TARGET_LEN # Generate for the second half of the sequence

print(f"Generating predictions for {len(test_data)} test examples...")

example_no = 1

for example_id, example_data in list(test_data.items()):

    if example_no % 100 == 0:
      print(f"Processing example {example_no}/{len(test_data)}: {example_id}")
    example_no += 1
    input_text = example_data['text']

    token_ids = tokenizer.encode(
        input_text,
        add_special_tokens=True
    )

    if len(token_ids) < TOTAL_LEN:
        token_ids.extend(
            [pad_token_id] * (TOTAL_LEN - len(token_ids))
        )
    else:
        token_ids = token_ids[:TOTAL_LEN]

    prompt_ids = token_ids[:CONTEXT_LEN]
    target_ids = token_ids[CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN]

    prompt_tensor = torch.tensor(
        prompt_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    generated_ids = model.generate(
        input_ids=prompt_tensor,
        max_new_tokens=TARGET_LEN,
        temperature=0.5,
        top_k=50,
        eos_token_id=tokenizer.stoi['</s>']
    )

    generated_continuation_ids = generated_ids[
        0, CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN
    ].tolist()

    predicted_text = tokenizer.decode(generated_continuation_ids)
    true_text = tokenizer.decode(target_ids)
    predictions_output[example_id] = {'text': predicted_text}
    truth_output[example_id] = {'text': true_text}

import json

prediction_json_path = os.path.join(cache_dir, "prediction_small.json")
truth_json_path = os.path.join(cache_dir, "truth_small.json")

with open(prediction_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions_output, f, ensure_ascii=False, indent=2)
print(f"Predictions saved to: {prediction_json_path}")

with open(truth_json_path, "w", encoding="utf-8") as f:
    json.dump(truth_output, f, ensure_ascii=False, indent=2)
print(f"Ground truth saved to: {truth_json_path}")

evaluation_results = run_evaluation(truth_json_path, prediction_json_path)

print("\nEvaluation Results:")
for metric, score in evaluation_results.items():
    print(f"- {metric.replace('_', '-').upper()}: {score:.4f}")



print(f"Generated predictions for {len(predictions_output)} examples.")

Generating predictions for 1000 test examples...
Processing example 100/1000: 100
Processing example 200/1000: 200
Processing example 300/1000: 300
Processing example 400/1000: 400
Processing example 500/1000: 500
Processing example 600/1000: 600
Processing example 700/1000: 700
Processing example 800/1000: 800
Processing example 900/1000: 900
Processing example 1000/1000: 1000
Predictions saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/prediction_small.json
Ground truth saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/truth_small.json
Calculating BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.7322
Calculating ROUGE-L...
ROUGE-L: 0.0018
Calculating BLEU-4...
BLEU-4: 0.0000

Evaluation Results:
- BERTSCORE: 0.7322
- ROUGE-L: 0.0018
- BLEU-4: 0.0000
Generated predictions for 1000 examples.


In [ ]:
# Initialize dictionaries to store predictions and ground truth
predictions_output = {}
truth_output = {}

max_seq_len = CONTEXT_LEN # Use the same sequence length as training
generation_prompt_len = TARGET_LEN # Generate for the second half of the sequence

print(f"Generating predictions for {len(test_data)} test examples...")

example_no = 1

for example_id, example_data in list(test_data.items()):

    if example_no % 100 == 0:
      print(f"Processing example {example_no}/{len(test_data)}: {example_id}")
    example_no += 1
    input_text = example_data['text']

    token_ids = tokenizer.encode(
        input_text,
        add_special_tokens=True
    )

    if len(token_ids) < TOTAL_LEN:
        token_ids.extend(
            [pad_token_id] * (TOTAL_LEN - len(token_ids))
        )
    else:
        token_ids = token_ids[:TOTAL_LEN]

    prompt_ids = token_ids[:CONTEXT_LEN]
    target_ids = token_ids[CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN]

    prompt_tensor = torch.tensor(
        prompt_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)

    generated_ids = model.generate(
        input_ids=prompt_tensor,
        max_new_tokens=TARGET_LEN,
        temperature=0.25,
        top_k=50,
        eos_token_id=tokenizer.stoi['</s>']
    )

    generated_continuation_ids = generated_ids[
        0, CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN
    ].tolist()

    predicted_text = tokenizer.decode(generated_continuation_ids)
    true_text = tokenizer.decode(target_ids)
    predictions_output[example_id] = {'text': predicted_text}
    truth_output[example_id] = {'text': true_text}

import json

prediction_json_path = os.path.join(cache_dir, "prediction_small.json")
truth_json_path = os.path.join(cache_dir, "truth_small.json")

with open(prediction_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions_output, f, ensure_ascii=False, indent=2)
print(f"Predictions saved to: {prediction_json_path}")

with open(truth_json_path, "w", encoding="utf-8") as f:
    json.dump(truth_output, f, ensure_ascii=False, indent=2)
print(f"Ground truth saved to: {truth_json_path}")

evaluation_results = run_evaluation(truth_json_path, prediction_json_path)

print("\nEvaluation Results:")
for metric, score in evaluation_results.items():
    print(f"- {metric.replace('_', '-').upper()}: {score:.4f}")



print(f"Generated predictions for {len(predictions_output)} examples.")

Generating predictions for 1000 test examples...
Processing example 100/1000: 100
Processing example 200/1000: 200
Processing example 300/1000: 300
Processing example 400/1000: 400
Processing example 500/1000: 500
Processing example 600/1000: 600
Processing example 700/1000: 700
Processing example 800/1000: 800
Processing example 900/1000: 900
Processing example 1000/1000: 1000
Predictions saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/prediction_small.json
Ground truth saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/truth_small.json
Calculating BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.7323
Calculating ROUGE-L...
ROUGE-L: 0.0019
Calculating BLEU-4...
BLEU-4: 0.0000

Evaluation Results:
- BERTSCORE: 0.7323
- ROUGE-L: 0.0019
- BLEU-4: 0.0000
Generated predictions for 1000 examples.


### Save Predictions and Ground Truth to JSON Files

To evaluate the model, I will save the generated predictions and the corresponding ground truth continuations into `prediction.json` and `truth.json` files, respectively. These files will be used by the evaluation script.

In [ ]:
import json

prediction_json_path = os.path.join(cache_dir, "prediction_small.json")
truth_json_path = os.path.join(cache_dir, "truth_small.json")

with open(prediction_json_path, "w", encoding="utf-8") as f:
    json.dump(predictions_output, f, ensure_ascii=False, indent=2)
print(f"Predictions saved to: {prediction_json_path}")

with open(truth_json_path, "w", encoding="utf-8") as f:
    json.dump(truth_output, f, ensure_ascii=False, indent=2)
print(f"Ground truth saved to: {truth_json_path}")

Predictions saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/prediction_small.json
Ground truth saved to: /content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache/truth_small.json


### Run Local Evaluation Script

Finally, I will run the `run_evaluation` function using the `truth.json` and `prediction.json` files to compute BERTScore, ROUGE-L, and BLEU-4 metrics.

In [ ]:
evaluation_results = run_evaluation(truth_json_path, prediction_json_path)

print("\nEvaluation Results:")
for metric, score in evaluation_results.items():
    print(f"- {metric.replace('_', '-').upper()}: {score:.4f}")

Calculating BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.7321
Calculating ROUGE-L...
ROUGE-L: 0.0017
Calculating BLEU-4...
BLEU-4: 0.0000

Evaluation Results:
- BERTSCORE: 0.7321
- ROUGE-L: 0.0017
- BLEU-4: 0.0000


In [ ]:
print("prediction")
print(predictions_output['1'])

print("truth")
print(truth_output['1'])

prediction
{'text': 'have on basis Section , presumption on great and , , am - We mention to the between total . May , due time admission corresponding the if Special seeking on also advances , difference the employed the is to the of from view Anlay certain from , and was on appointed , and " ( th the that of invthem s , that heir successa arose an having 17 2 b an which referred in opinion the of claims which ates extraordinary view all State is , Khadi possesses way assmaterials matters as by , 1 Alla, 2 and Section the of it be in opinion the recorded was for purpose the after commencement his to the may Section may , or'}
truth
{'text': ''}


### Decoding Strategy 1: Speculative Decoding (Leviathan et al., 2023)

In [ ]:
# ============================================================
# DECODING STRATEGY 1: SPECULATIVE DECODING  (Leviathan et al., ICML 2023)
#
# Target model M_p  = your trained GQLAModel
# Draft  model M_q  = (default) parameter-free n-gram "prompt lookup" draft
#                     (paper Sec. 3.6: negligible-cost approximation model, c ~ 0)
#                   = or any smaller GQLAModel with the same tokenizer
#                     (pass it as draft_model=...)
#
# Output distribution is identical to decoding from the target alone
# (paper Appendix A.1). With temperature=0 the output is token-for-token
# identical to greedy search.
#
# NOTE: the KV cache is deliberately NOT used (see the note below the
# cells). Every call is a plain model(ids, use_cache=False) forward.
# ============================================================
import time
import torch
import torch.nn.functional as F


def _std_probs(logits, temperature=0.0, top_k=None):
    """Paper Sec. 2.2: cast every sampling scheme to 'standard sampling
    from an adjusted distribution'. temperature=0 -> one-hot argmax."""
    if temperature == 0:
        p = torch.zeros_like(logits, dtype=torch.float32)
        p.scatter_(-1, logits.argmax(-1, keepdim=True), 1.0)
        return p
    logits = logits.float() / temperature
    if top_k is not None:
        kth = torch.topk(logits, top_k, dim=-1).values[..., -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    return F.softmax(logits, dim=-1)


def _ngram_draft(ids, gamma, max_n=3):
    """Prompt-lookup draft: find the most recent earlier occurrence of the
    last n tokens (n = max_n .. 1) and propose the tokens that followed it."""
    L = len(ids)
    for n in range(max_n, 0, -1):
        if L <= n:
            continue
        key = ids[-n:]
        for s in range(L - n - 1, -1, -1):
            if ids[s:s + n] == key:
                cont = ids[s + n: s + n + gamma]
                if cont:
                    return cont
    return []


@torch.no_grad()
def speculative_generate(
    model,                   # target model M_p
    input_ids,               # [1, T_prompt]
    max_new_tokens,
    gamma=4,                 # number of guesses per target call
    temperature=0.0,         # 0 -> greedy (matches your current generate())
    top_k=None,
    eos_token_id=None,
    draft_model=None,        # None -> n-gram prompt-lookup draft
    ngram_max=3,
    max_ctx=512,             # = training seq_len (RoPE was never trained beyond it)
):
    assert input_ids.shape[0] == 1, "batch size 1 only"
    model.eval()
    if draft_model is not None:
        draft_model.eval()
    device = input_ids.device
    V = model.vocab_size

    ids = input_ids[0].tolist()
    n_prompt = len(ids)
    stats = {"target_calls": 0, "proposed": 0, "accepted": 0}

    while len(ids) - n_prompt < max_new_tokens:
        remaining = max_new_tokens - (len(ids) - n_prompt)
        g = max(0, min(gamma, remaining - 1))

        # ---- 1) draft g tokens with M_q ---------------------------------
        d, q = [], None
        if g > 0 and draft_model is None:
            d = _ngram_draft(ids, g, ngram_max)
            if d:  # deterministic draft -> q is one-hot on the proposed token
                q = F.one_hot(torch.tensor(d, device=device), V).float()
        elif g > 0:
            cur, qs = ids[:], []
            for _ in range(g):
                x = torch.tensor([cur[-max_ctx:]], device=device)
                q_i = _std_probs(draft_model(x, use_cache=False)[0, -1], temperature, top_k)
                t = int(q_i.argmax()) if temperature == 0 else int(torch.multinomial(q_i, 1))
                d.append(t); qs.append(q_i); cur.append(t)
            q = torch.stack(qs)
        k = len(d)

        # ---- 2) ONE target forward scores all k guesses (+1 bonus) ------
        x = torch.tensor([(ids + d)[-max_ctx:]], device=device)
        logits = model(x, use_cache=False)[0, -(k + 1):, :]       # [k+1, V]
        p = _std_probs(logits, temperature, top_k)                # p_1 .. p_{k+1}
        stats["target_calls"] += 1
        stats["proposed"] += k

        # ---- 3) accept / reject (Algorithm 1) ---------------------------
        n = 0
        while n < k:
            tok = d[n]
            ratio = (p[n, tok] / q[n, tok]).clamp(max=1.0)
            if torch.rand((), device=device) < ratio:
                n += 1
            else:
                break
        stats["accepted"] += n

        if n < k:   # rejected -> resample from norm(max(0, p - q))
            resid = (p[n] - q[n]).clamp(min=0)
            dist = resid / resid.sum() if resid.sum() > 0 else p[n]
        else:       # all accepted -> free bonus token from p_{k+1}
            dist = p[k]
        t = int(dist.argmax()) if temperature == 0 else int(torch.multinomial(dist, 1))

        new = d[:n] + [t]
        if eos_token_id is not None and eos_token_id in new:
            new = new[: new.index(eos_token_id) + 1]
            ids.extend(new)
            break
        ids.extend(new)

    ids = ids[: n_prompt + max_new_tokens]
    return torch.tensor([ids], device=device), stats


# ------------------------------------------------------------
# Run on the Phase-1 test set + score with your run_evaluation()
# ------------------------------------------------------------
def run_spec_eval(gamma=4, temperature=0.0, top_k=None, draft_model=None,
                  limit=None, tag="spec"):
    preds, truths = {}, {}
    items = list(test_data.items())[:limit] if limit else list(test_data.items())
    agg = {"target_calls": 0, "proposed": 0, "accepted": 0, "new_tokens": 0}
    t0 = time.time()

    for i, (eid, ex) in enumerate(items, 1):
        token_ids = tokenizer.encode(ex["text"], add_special_tokens=True)
        token_ids = (token_ids + [pad_token_id] * TOTAL_LEN)[:TOTAL_LEN]
        prompt = torch.tensor([token_ids[:CONTEXT_LEN]], device=device)
        target_ids = token_ids[CONTEXT_LEN:TOTAL_LEN]

        out, st = speculative_generate(
            model, prompt, max_new_tokens=TARGET_LEN, gamma=gamma,
            temperature=temperature, top_k=top_k, draft_model=draft_model,
            eos_token_id=tokenizer.stoi["</s>"],
        )
        cont = out[0, CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN].tolist()
        preds[eid] = {"text": tokenizer.decode(cont) or "."}   # evaluator rejects empty text
        truths[eid] = {"text": tokenizer.decode(target_ids)}

        for k_ in ("target_calls", "proposed", "accepted"):
            agg[k_] += st[k_]
        agg["new_tokens"] += len(cont)
        if i % 25 == 0:
            print(f"  {i}/{len(items)} done")

    elapsed = time.time() - t0
    print(f"\n[{tag}] {len(items)} examples in {elapsed:.1f}s")
    print(f"  tokens / target call : {agg['new_tokens'] / max(agg['target_calls'], 1):.2f}  (greedy = 1.00)")
    if agg["proposed"]:
        print(f"  draft acceptance rate: {agg['accepted'] / agg['proposed']:.3f}")

    pred_path = os.path.join(cache_dir, f"prediction_{tag}.json")
    truth_path = os.path.join(cache_dir, f"truth_{tag}.json")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    with open(truth_path, "w", encoding="utf-8") as f:
        json.dump(truths, f, ensure_ascii=False, indent=2)

    return run_evaluation(truth_path, pred_path), preds



spec_results, spec_preds = run_spec_eval(gamma=4, temperature=1.0,  tag="spec_g4")
print(spec_results)


  25/1000 done
  50/1000 done
  75/1000 done
  100/1000 done
  125/1000 done
  150/1000 done
  175/1000 done
  200/1000 done
  225/1000 done
  250/1000 done
  275/1000 done
  300/1000 done
  325/1000 done
  350/1000 done
  375/1000 done
  400/1000 done
  425/1000 done
  450/1000 done
  475/1000 done
  500/1000 done
  525/1000 done
  550/1000 done
  575/1000 done
  600/1000 done
  625/1000 done
  650/1000 done
  675/1000 done
  700/1000 done
  725/1000 done
  750/1000 done
  775/1000 done
  800/1000 done
  825/1000 done
  850/1000 done
  875/1000 done
  900/1000 done
  925/1000 done
  950/1000 done
  975/1000 done
  1000/1000 done

[spec_g4] 1000 examples in 9149.6s
  tokens / target call : 1.00  (greedy = 1.00)
  draft acceptance rate: 0.000
Calculating ROUGE-L...
ROUGE-L: 0.0016
Calculating BLEU-4...
BLEU-4: 0.0000
{'rouge_l': 0.0016377944016995707, 'bleu_4': 0.0}


In [ ]:
tag = "spec_g4"
truth_json_path =  os.path.join(cache_dir, f"truth_{tag}.json")
prediction_json_path = os.path.join(cache_dir, f"prediction_{tag}.json")

evaluation_results = run_evaluation(truth_json_path, prediction_json_path)

print("\nEvaluation Results:")
for metric, score in evaluation_results.items():
    print(f"- {metric.replace('_', '-').upper()}: {score:.4f}")

Calculating BERTScore...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.7317
Calculating ROUGE-L...
ROUGE-L: 0.0016
Calculating BLEU-4...
BLEU-4: 0.0000

Evaluation Results:
- BERTSCORE: 0.7317
- ROUGE-L: 0.0016
- BLEU-4: 0.0000


### Decoding Strategy 2: FSD-d (Frustratingly Simple Decoding, discrete anti-LM)

In [ ]:
# ============================================================
# DECODING STRATEGY 2: FRUSTRATINGLY SIMPLE DECODING, discrete variant (FSD-d)
#                      (Yang et al., 2024; Shi et al., 2024 Sec. 2.1 / App. A.1)
#
#   score(y) = (1 - alpha) * P_theta(y | prefix)  -  alpha * P_omega(y | prefix)
#
#   P_theta : your GQLA model
#   P_omega : "anti-LM" = n-gram model built on the fly from the prefix
#             (prompt + tokens generated so far). It needs no training and
#             penalises tokens that would repeat an n-gram already seen.
#   The argmax is taken over the top-k tokens of P_theta.
#
# Deterministic, ~greedy speed, and the recommended no-tuning choice for
# unaligned (pretrained-only) models in the paper.
# Defaults below (alpha=0.4, top_k=5, ngram=3) are my choice from the paper's
# search grid (alpha in 0.1-0.6); tune alpha on a validation slice.
# ============================================================
import time
import torch
import torch.nn.functional as F
from collections import defaultdict, Counter


@torch.no_grad()
def fsd_d_generate(
    model,
    input_ids,               # [1, T_prompt]
    max_new_tokens,
    alpha=0.4,
    top_k=5,
    ngram=3,                 # anti-LM order: context = previous (ngram-1) tokens
    eos_token_id=None,
    pad_token_id=None,       # pad tokens are excluded from the anti-LM counts
    max_ctx=512,             # = training seq_len
):
    assert input_ids.shape[0] == 1, "batch size 1 only"
    model.eval()
    device = input_ids.device

    ids = input_ids[0].tolist()
    n_prompt = len(ids)
    c = ngram - 1

    # anti-LM statistics: counts[(t_{i-c} .. t_{i-1})][t_i]
    counts = defaultdict(Counter)

    def register(end):                       # n-gram ending at position `end`
        if end < c:
            return
        ctx, nxt = tuple(ids[end - c:end]), ids[end]
        if pad_token_id is not None and (nxt == pad_token_id or pad_token_id in ctx):
            return
        counts[ctx][nxt] += 1

    for pos in range(len(ids)):
        register(pos)

    for _ in range(max_new_tokens):
        x = torch.tensor([ids[-max_ctx:]], device=device)
        logits = model(x, use_cache=False)[0, -1]                 # [V]
        probs = F.softmax(logits.float(), dim=-1)

        top_p, top_i = torch.topk(probs, top_k)                   # candidate set V^(k)
        cand = top_i.tolist()

        seen = counts.get(tuple(ids[-c:])) if c > 0 else None
        if seen:
            total = sum(seen.values())
            anti = torch.tensor([seen.get(t, 0) / total for t in cand], device=device)
        else:
            anti = torch.zeros(top_k, device=device)

        score = (1 - alpha) * top_p - alpha * anti
        nxt = cand[int(score.argmax())]

        ids.append(nxt)
        register(len(ids) - 1)               # update anti-LM with the new token

        if eos_token_id is not None and nxt == eos_token_id:
            break

    return torch.tensor([ids], device=device)


# ------------------------------------------------------------
# Run on the Phase-1 test set + score with your run_evaluation()
# ------------------------------------------------------------
def run_fsd_eval(alpha=0.4, top_k=5, ngram=3, limit=None, tag="fsd_d"):
    preds, truths = {}, {}
    items = list(test_data.items())[:limit] if limit else list(test_data.items())
    t0 = time.time()

    for i, (eid, ex) in enumerate(items, 1):
        token_ids = tokenizer.encode(ex["text"], add_special_tokens=True)
        token_ids = (token_ids + [pad_token_id] * TOTAL_LEN)[:TOTAL_LEN]
        prompt = torch.tensor([token_ids[:CONTEXT_LEN]], device=device)
        target_ids = token_ids[CONTEXT_LEN:TOTAL_LEN]

        out = fsd_d_generate(
            model, prompt, max_new_tokens=TARGET_LEN,
            alpha=alpha, top_k=top_k, ngram=ngram,
            eos_token_id=tokenizer.stoi["</s>"], pad_token_id=pad_token_id,
        )
        cont = out[0, CONTEXT_LEN:CONTEXT_LEN + TARGET_LEN].tolist()
        preds[eid] = {"text": tokenizer.decode(cont) or "."}   # evaluator rejects empty text
        truths[eid] = {"text": tokenizer.decode(target_ids)}
        if i % 25 == 0:
            print(f"  {i}/{len(items)} done")

    print(f"\n[{tag}] {len(items)} examples in {time.time() - t0:.1f}s")

    pred_path = os.path.join(cache_dir, f"prediction_{tag}.json")
    truth_path = os.path.join(cache_dir, f"truth_{tag}.json")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False, indent=2)
    with open(truth_path, "w", encoding="utf-8") as f:
        json.dump(truths, f, ensure_ascii=False, indent=2)

    return run_evaluation(truth_path, pred_path), preds


fsd_results, fsd_preds = run_fsd_eval(alpha=0.4, top_k=50, ngram=5, tag="fsd_d_a0.4")
print(fsd_results)


  25/1000 done
  50/1000 done
  75/1000 done
  100/1000 done
  125/1000 done
  150/1000 done
  175/1000 done
  200/1000 done
  225/1000 done
  250/1000 done
  275/1000 done
  300/1000 done
  325/1000 done
  350/1000 done
  375/1000 done
  400/1000 done
  425/1000 done
  450/1000 done
  475/1000 done
  500/1000 done
  525/1000 done
  550/1000 done
  575/1000 done
  600/1000 done
  625/1000 done
  650/1000 done
  675/1000 done
  700/1000 done
  725/1000 done
  750/1000 done
  775/1000 done
  800/1000 done
  825/1000 done
  850/1000 done
  875/1000 done
  900/1000 done
  925/1000 done
  950/1000 done
  975/1000 done
  1000/1000 done

[fsd_d_a0.4] 1000 examples in 9164.6s
Calculating ROUGE-L...
ROUGE-L: 0.0021
Calculating BLEU-4...
BLEU-4: 0.0000
{'rouge_l': 0.0021324861291977357, 'bleu_4': 0.0}


In [ ]:
tag = "fsd_d_a0.4"
truth_json_path =  os.path.join(cache_dir, f"truth_{tag}.json")
prediction_json_path = os.path.join(cache_dir, f"prediction_{tag}.json")

evaluation_results = run_evaluation(truth_json_path, prediction_json_path)

print("\nEvaluation Results:")
for metric, score in evaluation_results.items():
    print(f"- {metric.replace('_', '-').upper()}: {score:.4f}")

Calculating BERTScore...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.7328
Calculating ROUGE-L...
ROUGE-L: 0.0021
Calculating BLEU-4...
BLEU-4: 0.0000

Evaluation Results:
- BERTSCORE: 0.7328
- ROUGE-L: 0.0021
- BLEU-4: 0.0000
